# **Tuning, Validación e Interpretabilidad**

In [9]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
from scipy.stats import randint, uniform

In [10]:
import sklearn
print(sklearn.__version__)

1.3.2


In [11]:
# =============================================================================
# PREPROCESAMIENTO CON PIPELINE
# =============================================================================

# Cargar datos
df = pd.read_pickle('data_limpia.pkl')

# Separar características y target
X = df.drop('credit_score', axis=1)
y = df['credit_score']

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [12]:

# Definir columnas
numerical_cols = ['age', 'monthly_inhand_salary', 'num_bank_accounts', 'num_credit_card', 
                 'interest_rate', 'delay_from_due_date', 'num_of_delayed_payment', 
                 'changed_credit_limit', 'num_credit_inquiries', 'outstanding_debt', 
                 'credit_utilization_ratio', 'credit_history_age', 'total_emi_per_month', 
                 'amount_invested_monthly', 'monthly_balance']

categorical_cols = ['occupation', 'credit_mix', 'payment_of_min_amount', 'payment_behaviour']

# Codificar target
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

print(f"✅ Datos preparados: {X_train.shape[0]} train, {X_test.shape[0]} test")


✅ Datos preparados: 70000 train, 30000 test


In [13]:
# =============================================================================
# PIPELINE COMPLETO CON PREPROCESAMIENTO
# =============================================================================

# Crear preprocesador
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_cols)
    ],
    remainder='passthrough'
)

# Obtener nombres de características después del preprocesamiento
preprocessor.fit(X_train)
feature_names = (list(numerical_cols) + 
                list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)) +
                ['not_specified','credit_builder_loan','personal_loan','debt_consolidation_loan',
                 'student_loan', 'payday_loan','mortgage_loan','auto_loan','home_equity_loan'])

print(f"📊 Dimensionalidad final: {len(feature_names)} características")

📊 Dimensionalidad final: 48 características


# **KNN**

In [14]:
# =============================================================================
# KNN AVANZADO OPTIMIZADO: VALIDACIÓN, TUNING E INTERPRETABILIDAD
# =============================================================================

print("🧠 INICIANDO KNN AVANZADO OPTIMIZADO - ...")
start_time = time.time()

# Configuraciones de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


# =============================================================================
# CONFIGURACIÓN RÁPIDA - OPTIMIZACIÓN CON RANDOMIZED SEARCH (PRINCIPAL)
# =============================================================================

print(f"\n🎯 INICIANDO OPTIMIZACIÓN RÁPIDA CON RANDOMIZED SEARCH...")

# Definir pipeline
knn_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('knn', KNeighborsClassifier(n_jobs=-1))
])

# Distribuciones optimizadas para Randomized Search (EVITAR 'brute')
param_dist = {
    'knn__n_neighbors': randint(3, 20),
    'knn__weights': ['uniform', 'distance'],
    'knn__algorithm': ['auto', 'ball_tree', 'kd_tree'],  # ELIMINADO 'brute'
    'knn__p': [1, 2],
    'knn__leaf_size': randint(20, 40)
}

# Configurar Stratified K-Fold con MENOS folds para velocidad
skf_fast = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)  # 3 folds en vez de 5

# Randomized Search CV optimizado
random_search = RandomizedSearchCV(
    knn_pipeline,
    param_dist,
    n_iter=25,  # REDUCIDO de 50 a 25
    cv=skf_fast,
    scoring='f1_macro',
    n_jobs=-1,
    random_state=42,
    verbose=1,
    return_train_score=True
)

print("🔍 Ejecutando Randomized Search Optimizado...")
random_start = time.time()
random_search.fit(X_train, y_train_encoded)
random_time = time.time() - random_start

print(f"✅ Randomized Search completado en {random_time:.2f}s")
print(f"🏆 Mejores parámetros: {random_search.best_params_}")
print(f"📈 Mejor score CV: {random_search.best_score_:.4f}")

# =============================================================================
# BÚSQUEDA MANUAL RÁPIDA (ALTERNATIVA)
# =============================================================================

print(f"\n🎯 INICIANDO BÚSQUEDA MANUAL RÁPIDA...")

def evaluate_knn_config_fast(n_neighbors, weights, algorithm, p, leaf_size):
    """Evalúa una configuración específica de KNN (versión rápida)"""
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('knn', KNeighborsClassifier(
            n_neighbors=n_neighbors,
            weights=weights,
            algorithm=algorithm,
            p=p,
            leaf_size=leaf_size,
            n_jobs=-1
        ))
    ])
    
    cv_scores = cross_val_score(
        model, X_train, y_train_encoded, 
        cv=skf_fast, scoring='f1_macro', n_jobs=-1
    )
    return cv_scores.mean()

# Búsqueda manual optimizada - solo combinaciones prometedoras
best_manual_score = 0
best_manual_params = {}
manual_results = []

print("🔍 Explorando combinaciones clave...")
# Solo probar combinaciones más prometedoras basadas en conocimiento de dominio
promising_combinations = [
    (5, 'uniform', 'auto', 2, 30),
    (7, 'distance', 'auto', 2, 30),
    (9, 'uniform', 'ball_tree', 1, 25),
    (11, 'distance', 'kd_tree', 2, 35),
    (15, 'uniform', 'auto', 2, 30),
]

for k, weight, algo, p_val, leaf in promising_combinations:
    score = evaluate_knn_config_fast(k, weight, algo, p_val, leaf)
    manual_results.append({
        'n_neighbors': k, 
        'weights': weight, 
        'algorithm': algo,
        'p': p_val,
        'leaf_size': leaf,
        'score': score
    })
    if score > best_manual_score:
        best_manual_score = score
        best_manual_params = {
            'n_neighbors': k, 
            'weights': weight, 
            'algorithm': algo,
            'p': p_val,
            'leaf_size': leaf
        }

print(f"✅ Búsqueda manual rápida completada")
print(f"🏆 Mejores parámetros manuales: {best_manual_params}")
print(f"📈 Mejor score manual: {best_manual_score:.4f}")

# Entrenar modelo manual
knn_manual = Pipeline([
    ('preprocessor', preprocessor),
    ('knn', KNeighborsClassifier(**best_manual_params, n_jobs=-1))
])
knn_manual.fit(X_train, y_train_encoded)

# =============================================================================
# COMPARACIÓN RÁPIDA DE MODELOS OPTIMIZADOS
# =============================================================================

print(f"\n🔍 COMPARANDO MODELOS OPTIMIZADOS...")

models = {
    'RandomSearch_KNN': random_search.best_estimator_,
    'ManualSearch_KNN': knn_manual
}

results_advanced = {}

for model_name, model in models.items():
    print(f"\n🎯 Evaluando {model_name}...")
    
    # Predicciones
    y_pred = model.predict(X_test)
    
    # Métricas
    accuracy = accuracy_score(y_test_encoded, y_pred)
    precision = precision_score(y_test_encoded, y_pred, average='macro')
    recall = recall_score(y_test_encoded, y_pred, average='macro')
    f1 = f1_score(y_test_encoded, y_pred, average='macro')
    
    # Validación cruzada rápida
    cv_scores = cross_val_score(
        model, X_train, y_train_encoded, 
        cv=skf_fast, scoring='f1_macro', n_jobs=-1
    )
    
    results_advanced[model_name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'y_pred': y_pred
    }
    
    print(f"  ✅ F1-Score: {f1:.4f}")
    print(f"  ✅ CV F1-Score: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Seleccionar mejor modelo
best_model_name = max(results_advanced.items(), key=lambda x: x[1]['f1'])[0]
best_model_advanced = results_advanced[best_model_name]['model']
best_results_advanced = results_advanced[best_model_name]

print(f"\n🏆 MEJOR MODELO AVANZADO: {best_model_name}")
print(f"📊 F1-Score: {best_results_advanced['f1']:.4f}")

# =============================================================================
# INTERPRETABILIDAD - PERMUTATION IMPORTANCE (MUESTREO PARA VELOCIDAD)
# =============================================================================

print(f"\n🔍 CALCULANDO PERMUTATION IMPORTANCE (MUESTREADO)...")

# Preparar datos preprocesados
X_test_processed = preprocessor.transform(X_test)

# Muestrear para mayor velocidad (usar solo 1000 muestras)
sample_size = min(1000, X_test_processed.shape[0])
sample_indices = np.random.choice(X_test_processed.shape[0], sample_size, replace=False)
X_test_sampled = X_test_processed[sample_indices]
y_test_sampled = y_test_encoded[sample_indices]

# Calcular permutation importance con muestreo
perm_start = time.time()
perm_importance = permutation_importance(
    best_model_advanced.named_steps['knn'],
    X_test_sampled,
    y_test_sampled,
    n_repeats=5,  # REDUCIDO de 10 a 5
    random_state=42,
    n_jobs=-1,
    scoring='f1_macro'
)
perm_time = time.time() - perm_start

print(f"✅ Permutation Importance calculado en {perm_time:.2f}s")

# Crear DataFrame con resultados
perm_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)

print("\n📊 TOP 10 CARACTERÍSTICAS MÁS IMPORTANTES (Permutation):")
print(perm_df.head(10).round(4))

# =============================================================================
# VISUALIZACIONES ESENCIALES
# =============================================================================

plt.figure(figsize=(15, 10))

# Gráfico 1: Comparación de modelos optimizados
plt.subplot(2, 2, 1)
model_names = list(results_advanced.keys())
f1_scores = [results_advanced[model]['f1'] for model in model_names]
colors = ['skyblue', 'lightcoral']

bars = plt.bar(model_names, f1_scores, color=colors, alpha=0.8)
plt.ylabel('F1-Score (Test)')
plt.title('Comparación de Modelos Optimizados\nF1-Score en Test')
plt.xticks(rotation=45)
for bar, score in zip(bars, f1_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{score:.3f}', ha='center', va='bottom')

# Gráfico 2: Permutation Importance
plt.subplot(2, 2, 2)
top_perm = perm_df.head(12)
plt.barh(range(len(top_perm)), top_perm['importance_mean'], 
         xerr=top_perm['importance_std'], alpha=0.7, capsize=5)
plt.yticks(range(len(top_perm)), top_perm['feature'])
plt.xlabel('Permutation Importance')
plt.title('Top 12 Características - Permutation Importance')
plt.grid(True, alpha=0.3)

# Gráfico 3: Matriz de confusión del mejor modelo
plt.subplot(2, 2, 3)
y_pred_best_advanced = best_results_advanced['y_pred']
cm_best_advanced = confusion_matrix(y_test_encoded, y_pred_best_advanced)
sns.heatmap(cm_best_advanced, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Matriz de Confusión - {best_model_name}')
plt.ylabel('Real')
plt.xlabel('Predicho')

# Gráfico 4: Comparación de métodos de búsqueda
plt.subplot(2, 2, 4)
search_methods = ['Random Search', 'Manual Search']
search_scores = [random_search.best_score_, best_manual_score]
search_times = [random_time, 0]

bars = plt.bar(search_methods, search_scores, alpha=0.7, 
               color=['lightgreen', 'orange'])
plt.ylabel('Mejor F1-Score CV')
plt.title('Comparación de Métodos de Búsqueda')
for bar, score in zip(bars, search_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{score:.3f}', ha='center', va='bottom')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# =============================================================================
# ANÁLISIS RÁPIDO DE COMPORTAMIENTO
# =============================================================================

print(f"\n📈 ANALIZANDO COMPORTAMIENTO DEL MODELO...")

# Usar resultados de Randomized Search para análisis
results_df = pd.DataFrame(random_search.cv_results_)
train_scores = results_df['mean_train_score']
test_scores = results_df['mean_test_score']
gap = train_scores - test_scores

print(f"📊 Análisis de sobreajuste:")
print(f"  • Diferencia promedio train-test: {gap.mean():.4f}")
print(f"  • Configuraciones probadas: {len(results_df)}")

# =============================================================================
# RESUMEN FINAL OPTIMIZADO
# =============================================================================

end_time = time.time()
execution_time = end_time - start_time

print("\n" + "="*80)
print("🎯 RESUMEN FINAL - KNN AVANZADO OPTIMIZADO (VERSIÓN RÁPIDA)")
print("="*80)

print(f"⏱️  Tiempo total ejecución: {execution_time:.2f}s")
print(f"🏆 Mejor modelo: {best_model_name}")
print(f"📈 F1-Score Test: {best_results_advanced['f1']:.4f} ({best_results_advanced['f1']*100:.2f}%)")
print(f"🎯 Validación Cruzada: {best_results_advanced['cv_mean']:.4f} ± {best_results_advanced['cv_std']*2:.4f}")

print(f"\n🔧 MEJORES HIPERPARÁMETROS:")
if best_model_name == 'RandomSearch_KNN':
    best_params = random_search.best_params_
else:
    best_params = best_manual_params

for param, value in best_params.items():
    param_name = param.replace('knn__', '')
    print(f"  {param_name}: {value}")

print(f"\n📊 CARACTERÍSTICAS MÁS IMPORTANTES:")
for i in range(5):
    feature = perm_df.iloc[i]['feature']
    importance = perm_df.iloc[i]['importance_mean']
    print(f"  {i+1}. {feature}: {importance:.4f}")

print(f"\n⚡ EFICIENCIA COMPUTACIONAL:")
print(f"  Randomized Search: {random_time:.2f}s (25 iteraciones, 3-folds)")
print(f"  Permutation Importance: {perm_time:.2f}s (5 repeticiones, 1000 muestras)")

print(f"\n📋 REPORTE CLASIFICACIÓN ({best_model_name}):")
print(classification_report(y_test_encoded, y_pred_best_advanced, target_names=le.classes_))

# =============================================================================
# GUARDADO DE RESULTADOS
# =============================================================================

advanced_results = {
    'best_model': best_model_advanced,
    'best_model_name': best_model_name,
    'best_params': best_params,
    'random_search_results': random_search.cv_results_,
    'manual_search_results': manual_results,
    'permutation_importance': perm_df,
    'feature_names': feature_names,
    'results_advanced': results_advanced,
    'execution_time': execution_time,
    'preprocessor': preprocessor,
    'label_encoder': le
}

# joblib.dump(advanced_results, 'knn_advanced_optimized.pkl')
# print(f"\n💾 Resultados optimizados guardados: 'knn_advanced_optimized.pkl'")

# Guardar el mejor modelo por separado
joblib.dump(best_model_advanced, 'best_knn_model_optimized.pkl')
print(f"💾 Mejor modelo guardado: 'best_knn_model_optimized.pkl'")

print("="*80)
print("✅ ANÁLISIS COMPLETADO - KNN OPTIMIZADO (VERSIÓN RÁPIDA)")
print("="*80)0o<o

🧠 INICIANDO KNN AVANZADO OPTIMIZADO - ...

🎯 INICIANDO OPTIMIZACIÓN RÁPIDA CON RANDOMIZED SEARCH...
🔍 Ejecutando Randomized Search Optimizado...
Fitting 3 folds for each of 25 candidates, totalling 75 fits
✅ Randomized Search completado en 17176.87s
🏆 Mejores parámetros: {'knn__algorithm': 'auto', 'knn__leaf_size': 20, 'knn__n_neighbors': 9, 'knn__p': 1, 'knn__weights': 'distance'}
📈 Mejor score CV: 0.7446

🎯 INICIANDO BÚSQUEDA MANUAL RÁPIDA...
🔍 Explorando combinaciones clave...


KeyboardInterrupt: 

# **Random Forest**

In [ ]:
# =============================================================================
# OPTIMIZACIÓN CON RANDOMIZED SEARCH CV (PRINCIPAL)
# =============================================================================

from sklearn.ensemble import RandomForestClassifier

print(f"\n🎯 INICIANDO OPTIMIZACIÓN CON RANDOMIZED SEARCH CV...")

# Definir pipeline
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestClassifier(random_state=42, n_jobs=-1))
])

# Distribuciones optimizadas para Randomized Search
param_dist = {
    'rf__n_estimators': randint(50, 200),
    'rf__max_depth': [10, 15, 20, None],
    'rf__min_samples_split': randint(2, 15),
    'rf__min_samples_leaf': randint(1, 8),
    'rf__max_features': ['sqrt', 'log2', 0.3],
    'rf__bootstrap': [True],
    'rf__class_weight': [None, 'balanced']
}

# Configurar Stratified K-Fold (MANTENIENDO 5 folds)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Randomized Search CV optimizado
random_search = RandomizedSearchCV(
    rf_pipeline,
    param_dist,
    n_iter=30,  # REDUCIDO de 50 a 30
    cv=skf,
    scoring='f1_macro',
    n_jobs=-1,
    random_state=42,
    verbose=1,
    return_train_score=True
)

print("🔍 Ejecutando Randomized Search Optimizado...")
random_start = time.time()
random_search.fit(X_train, y_train_encoded)
random_time = time.time() - random_start

print(f"✅ Randomized Search completado en {random_time:.2f}s")
print(f"🏆 Mejores parámetros: {random_search.best_params_}")
print(f"📈 Mejor score CV: {random_search.best_score_:.4f}")

# =============================================================================
# BÚSQUEDA MANUAL RÁPIDA (ALTERNATIVA)
# =============================================================================

print(f"\n🎯 INICIANDO BÚSQUEDA MANUAL RÁPIDA...")

def evaluate_rf_config_fast(n_estimators, max_depth, min_samples_split, min_samples_leaf, 
                           max_features, class_weight):
    """Evalúa una configuración específica de Random Forest (versión rápida)"""
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('rf', RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            bootstrap=True,
            class_weight=class_weight,
            random_state=42,
            n_jobs=-1
        ))
    ])
    
    cv_scores = cross_val_score(
        model, X_train, y_train_encoded, 
        cv=skf, scoring='f1_macro', n_jobs=-1
    )
    return cv_scores.mean()

# Búsqueda manual optimizada - solo combinaciones prometedoras
best_manual_score = 0
best_manual_params = {}
manual_results = []

print("🔍 Explorando combinaciones clave de Random Forest...")
# Combinaciones basadas en mejores prácticas para RF
promising_combinations = [
    (100, 15, 2, 1, 'sqrt', None),
    (150, 20, 5, 2, 'log2', None),
    (200, None, 10, 4, 'sqrt', 'balanced'),
    (100, 10, 2, 1, 0.3, None),
    (200, 15, 5, 2, 'log2', 'balanced'),
]

for n_est, max_d, min_split, min_leaf, max_feat, class_w in promising_combinations:
    score = evaluate_rf_config_fast(n_est, max_d, min_split, min_leaf, max_feat, class_w)
    manual_results.append({
        'n_estimators': n_est, 
        'max_depth': max_d, 
        'min_samples_split': min_split,
        'min_samples_leaf': min_leaf,
        'max_features': max_feat,
        'class_weight': class_w,
        'score': score
    })
    if score > best_manual_score:
        best_manual_score = score
        best_manual_params = {
            'n_estimators': n_est, 
            'max_depth': max_d, 
            'min_samples_split': min_split,
            'min_samples_leaf': min_leaf,
            'max_features': max_feat,
            'bootstrap': True,
            'class_weight': class_w
        }

print(f"✅ Búsqueda manual rápida completada")
print(f"🏆 Mejores parámetros manuales: {best_manual_params}")
print(f"📈 Mejor score manual: {best_manual_score:.4f}")

# Entrenar modelo manual
rf_manual = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestClassifier(**best_manual_params, random_state=42, n_jobs=-1))
])
rf_manual.fit(X_train, y_train_encoded)

# =============================================================================
# COMPARACIÓN RÁPIDA DE MODELOS OPTIMIZADOS
# =============================================================================

print(f"\n🔍 COMPARANDO MODELOS OPTIMIZADOS...")

models = {
    'RandomSearch_RF': random_search.best_estimator_,
    'ManualSearch_RF': rf_manual
}

results_advanced = {}

for model_name, model in models.items():
    print(f"\n🎯 Evaluando {model_name}...")
    
    # Predicciones
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    # Métricas
    accuracy = accuracy_score(y_test_encoded, y_pred)
    precision = precision_score(y_test_encoded, y_pred, average='macro')
    recall = recall_score(y_test_encoded, y_pred, average='macro')
    f1 = f1_score(y_test_encoded, y_pred, average='macro')
    
    # Validación cruzada completa
    cv_scores = cross_val_score(
        model, X_train, y_train_encoded, 
        cv=skf, scoring='f1_macro', n_jobs=-1
    )
    
    # Importancia de características
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': model.named_steps['rf'].feature_importances_
    }).sort_values('importance', ascending=False)
    
    results_advanced[model_name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'feature_importance': feature_importance,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f"  ✅ F1-Score: {f1:.4f}")
    print(f"  ✅ CV F1-Score: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Seleccionar mejor modelo
best_model_name = max(results_advanced.items(), key=lambda x: x[1]['f1'])[0]
best_model_advanced = results_advanced[best_model_name]['model']
best_results_advanced = results_advanced[best_model_name]

print(f"\n🏆 MEJOR MODELO AVANZADO: {best_model_name}")
print(f"📊 F1-Score: {best_results_advanced['f1']:.4f}")

# =============================================================================
# INTERPRETABILIDAD - PERMUTATION IMPORTANCE (OPTIMIZADO)
# =============================================================================

print(f"\n🔍 CALCULANDO PERMUTATION IMPORTANCE (OPTIMIZADO)...")

# Preparar datos preprocesados
X_test_processed = preprocessor.transform(X_test)

# Muestrear para mayor velocidad (usar solo 2000 muestras)
sample_size = min(2000, X_test_processed.shape[0])
sample_indices = np.random.choice(X_test_processed.shape[0], sample_size, replace=False)
X_test_sampled = X_test_processed[sample_indices]
y_test_sampled = y_test_encoded[sample_indices]

# Calcular permutation importance con muestreo
perm_start = time.time()
perm_importance = permutation_importance(
    best_model_advanced.named_steps['rf'],
    X_test_sampled,
    y_test_sampled,
    n_repeats=5,  # REDUCIDO de 10 a 5
    random_state=42,
    n_jobs=-1,
    scoring='f1_macro'
)
perm_time = time.time() - perm_start

print(f"✅ Permutation Importance calculado en {perm_time:.2f}s")

# Crear DataFrame con resultados
perm_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)

print("\n📊 TOP 10 CARACTERÍSTICAS MÁS IMPORTANTES (Permutation):")
print(perm_df.head(10).round(4))

# =============================================================================
# ANÁLISIS DE ESTABILIDAD RÁPIDO
# =============================================================================

print(f"\n📊 ANALIZANDO ESTABILIDAD ENTRE MÉTODOS DE OPTIMIZACIÓN...")

# Calcular importancia promedio entre métodos
all_methods_importance = []
for model_name in results_advanced.keys():
    importance_df = results_advanced[model_name]['feature_importance'].set_index('feature')['importance']
    all_methods_importance.append(importance_df)

stability_df = pd.DataFrame(all_methods_importance).T
stability_df.columns = list(results_advanced.keys())
stability_df['mean_importance'] = stability_df.mean(axis=1)
stability_df['std_importance'] = stability_df.std(axis=1)

print("Características más estables entre métodos de optimización:")
stable_features = stability_df.nsmallest(8, 'std_importance')['mean_importance']
print(stable_features.round(4))

# =============================================================================
# VISUALIZACIONES ESENCIALES
# =============================================================================

plt.figure(figsize=(20, 12))

# Gráfico 1: Comparación de modelos optimizados
plt.subplot(2, 3, 1)
model_names = list(results_advanced.keys())
f1_scores = [results_advanced[model]['f1'] for model in model_names]
colors = ['skyblue', 'lightcoral']

bars = plt.bar(model_names, f1_scores, color=colors, alpha=0.8)
plt.ylabel('F1-Score (Test)')
plt.title('Comparación de Modelos Optimizados\nF1-Score en Test')
plt.xticks(rotation=45)
for bar, score in zip(bars, f1_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{score:.3f}', ha='center', va='bottom')

# Gráfico 2: Permutation Importance
plt.subplot(2, 3, 2)
top_perm = perm_df.head(12)
plt.barh(range(len(top_perm)), top_perm['importance_mean'], 
         xerr=top_perm['importance_std'], alpha=0.7, capsize=5)
plt.yticks(range(len(top_perm)), top_perm['feature'])
plt.xlabel('Permutation Importance')
plt.title('Top 12 Características - Permutation Importance')
plt.grid(True, alpha=0.3)

# Gráfico 3: Importancia Gini del mejor modelo
plt.subplot(2, 3, 3)
top_gini = best_results_advanced['feature_importance'].head(12)
plt.barh(range(len(top_gini)), top_gini['importance'], alpha=0.7)
plt.yticks(range(len(top_gini)), top_gini['feature'])
plt.xlabel('Gini Importance')
plt.title('Top 12 Características - Gini Importance')
plt.grid(True, alpha=0.3)

# Gráfico 4: Matriz de confusión del mejor modelo
plt.subplot(2, 3, 4)
y_pred_best = best_results_advanced['y_pred']
cm_best = confusion_matrix(y_test_encoded, y_pred_best)
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Matriz de Confusión\n{best_model_name}')
plt.ylabel('Real')
plt.xlabel('Predicho')

# Gráfico 5: Comparación de métodos de búsqueda
plt.subplot(2, 3, 5)
search_methods = ['Random Search', 'Manual Search']
search_scores = [random_search.best_score_, best_manual_score]
search_times = [random_time, 0]

bars = plt.bar(search_methods, search_scores, alpha=0.7, 
               color=['lightgreen', 'orange'])
plt.ylabel('Mejor F1-Score CV')
plt.title('Comparación de Métodos de Búsqueda')
for bar, score in zip(bars, search_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{score:.3f}', ha='center', va='bottom')
plt.grid(True, alpha=0.3)

# Gráfico 6: Curva de importancia acumulativa
plt.subplot(2, 3, 6)
cumulative_importance = best_results_advanced['feature_importance']['importance'].cumsum()
plt.plot(range(1, len(cumulative_importance) + 1), cumulative_importance, linewidth=2)
plt.axhline(y=0.8, color='r', linestyle='--', alpha=0.7, label='80% importancia')
plt.axhline(y=0.9, color='g', linestyle='--', alpha=0.7, label='90% importancia')
plt.xlabel('Número de Características')
plt.ylabel('Importancia Acumulativa')
plt.title('Importancia Acumulativa de Características')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# =============================================================================
# ANÁLISIS RÁPIDO DE COMPORTAMIENTO
# =============================================================================

print(f"\n📈 ANALIZANDO COMPORTAMIENTO DEL MODELO...")

# Usar resultados de Randomized Search para análisis
results_df = pd.DataFrame(random_search.cv_results_)
train_scores = results_df['mean_train_score']
test_scores = results_df['mean_test_score']
gap = train_scores - test_scores

print(f"📊 Análisis de sobreajuste:")
print(f"  • Diferencia promedio train-test: {gap.mean():.4f}")
print(f"  • Configuraciones probadas: {len(results_df)}")
print(f"  • Mejor configuración: {random_search.best_score_:.4f}")

# =============================================================================
# RESUMEN FINAL OPTIMIZADO
# =============================================================================

end_time = time.time()
execution_time = end_time - start_time

print("\n" + "="*80)
print("🎯 RESUMEN FINAL - RANDOM FOREST AVANZADO OPTIMIZADO")
print("="*80)

print(f"⏱️  Tiempo total ejecución: {execution_time:.2f}s")
print(f"🏆 Mejor modelo: {best_model_name}")
print(f"📈 F1-Score Test: {best_results_advanced['f1']:.4f} ({best_results_advanced['f1']*100:.2f}%)")
print(f"🎯 Validación Cruzada: {best_results_advanced['cv_mean']:.4f} ± {best_results_advanced['cv_std']*2:.4f}")

print(f"\n🔧 MEJORES HIPERPARÁMETROS:")
if best_model_name == 'RandomSearch_RF':
    best_params = random_search.best_params_
else:
    best_params = best_manual_params

for param, value in best_params.items():
    param_name = param.replace('rf__', '')
    print(f"  {param_name}: {value}")

print(f"\n📊 CARACTERÍSTICAS MÁS IMPORTANTES:")
for i in range(5):
    feature = perm_df.iloc[i]['feature']
    importance = perm_df.iloc[i]['importance_mean']
    print(f"  {i+1}. {feature}: {importance:.4f}")

print(f"\n⚡ EFICIENCIA COMPUTACIONAL:")
print(f"  Randomized Search: {random_time:.2f}s (30 iteraciones, 5-folds)")
print(f"  Permutation Importance: {perm_time:.2f}s (5 repeticiones, 2000 muestras)")

print(f"\n🎯 ESTABILIDAD DEL MODELO:")
print(f"  Característica más estable: {stable_features.index[0]}")
print(f"  Número de características con >80% importancia: {(cumulative_importance <= 0.8).sum()}")

print(f"\n📋 REPORTE CLASIFICACIÓN ({best_model_name}):")
print(classification_report(y_test_encoded, y_pred_best, target_names=le.classes_))

# =============================================================================
# GUARDADO DE RESULTADOS
# =============================================================================

advanced_results = {
    'best_model': best_model_advanced,
    'best_model_name': best_model_name,
    'best_params': best_params,
    'random_search_results': random_search.cv_results_,
    'manual_search_results': manual_results,
    'permutation_importance': perm_df,
    'stability_analysis': stability_df,
    'feature_names': feature_names,
    'results_advanced': results_advanced,
    'execution_time': execution_time,
    'preprocessor': preprocessor,
    'label_encoder': le
}

# joblib.dump(advanced_results, 'random_forest_advanced_optimized.pkl')
# print(f"\n💾 Resultados optimizados guardados: 'random_forest_advanced_optimized.pkl'")

# Guardar el mejor modelo por separado
joblib.dump(best_model_advanced, 'best_rf_model_optimized.pkl')
print(f"💾 Mejor modelo guardado: 'best_rf_model_optimized.pkl'")

print("="*80)
print("✅ ANÁLISIS COMPLETADO - RANDOM FOREST OPTIMIZADO (VERSIÓN RÁPIDA)")
print("="*80)


🎯 INICIANDO OPTIMIZACIÓN CON RANDOMIZED SEARCH CV...


NameError: name 'RandomForestClassifier' is not defined

# **XGBoost**

In [ ]:
# OPTIMIZACIÓN CON RANDOMIZED SEARCH CV (PRINCIPAL)
# =============================================================================
import xgboost as xgb
print(f"\n🎯 INICIANDO OPTIMIZACIÓN CON RANDOMIZED SEARCH CV...")

# Definir pipeline
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('xgb', xgb.XGBClassifier(
        random_state=42,
        n_jobs=-1,
        tree_method='hist',  # Más rápido que 'exact'
        eval_metric='mlogloss'
    ))
])

# Distribuciones optimizadas para Randomized Search
param_dist = {
    'xgb__n_estimators': randint(50, 200),
    'xgb__max_depth': randint(3, 10),
    'xgb__learning_rate': uniform(0.01, 0.3),
    'xgb__subsample': uniform(0.6, 0.4),  # 0.6 a 1.0
    'xgb__colsample_bytree': uniform(0.6, 0.4),
    'xgb__reg_alpha': uniform(0, 1),
    'xgb__reg_lambda': uniform(1, 2),
    'xgb__min_child_weight': randint(1, 10)
}

# Configurar Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Randomized Search CV optimizado
random_search = RandomizedSearchCV(
    xgb_pipeline,
    param_dist,
    n_iter=25,  # REDUCIDO para mayor velocidad
    cv=skf,
    scoring='f1_macro',
    n_jobs=-1,
    random_state=42,
    verbose=1,
    return_train_score=True
)

print("🔍 Ejecutando Randomized Search Optimizado...")
random_start = time.time()
random_search.fit(X_train, y_train_encoded)
random_time = time.time() - random_start

print(f"✅ Randomized Search completado en {random_time:.2f}s")
print(f"🏆 Mejores parámetros: {random_search.best_params_}")
print(f"📈 Mejor score CV: {random_search.best_score_:.4f}")

# =============================================================================
# BÚSQUEDA MANUAL RÁPIDA SIMPLIFICADA (SIN EARLY STOPPING EN CV)
# =============================================================================

print(f"\n🎯 INICIANDO BÚSQUEDA MANUAL RÁPIDA...")

def evaluate_xgb_config_simple(n_estimators, max_depth, learning_rate, subsample, 
                              colsample_bytree, reg_alpha, reg_lambda, min_child_weight):
    """Evalúa una configuración específica de XGBoost sin early stopping en CV"""
    
    model = Pipeline([
        ('preprocessor', preprocessor),
        ('xgb', xgb.XGBClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            min_child_weight=min_child_weight,
            random_state=42,
            n_jobs=-1,
            tree_method='hist',
            eval_metric='mlogloss'
        ))
    ])
    
    # Usar validación cruzada simple sin early stopping
    cv_scores = cross_val_score(
        model, X_train, y_train_encoded, 
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),  # 3-folds para velocidad
        scoring='f1_macro', 
        n_jobs=-1
    )
    
    return cv_scores.mean()

# Búsqueda manual optimizada - combinaciones prometedoras para XGBoost
best_manual_score = 0
best_manual_params = {}
manual_results = []

print("🔍 Explorando combinaciones clave de XGBoost...")
# Combinaciones basadas en mejores prácticas para XGBoost
promising_combinations = [
    (100, 6, 0.1, 0.8, 0.8, 0.1, 1.0, 1),
    (150, 8, 0.05, 0.9, 0.7, 0.5, 1.5, 3),
    (200, 4, 0.2, 0.7, 0.9, 0.01, 2.0, 1),
    (100, 5, 0.15, 0.85, 0.75, 0.2, 1.2, 2),
    (120, 7, 0.08, 0.8, 0.8, 0.3, 1.8, 4),
]

for n_est, max_d, lr, subsample, colsample, alpha, lambda_, min_child in promising_combinations:
    try:
        score = evaluate_xgb_config_simple(n_est, max_d, lr, subsample, colsample, alpha, lambda_, min_child)
        manual_results.append({
            'n_estimators': n_est, 
            'max_depth': max_d, 
            'learning_rate': lr,
            'subsample': subsample,
            'colsample_bytree': colsample,
            'reg_alpha': alpha,
            'reg_lambda': lambda_,
            'min_child_weight': min_child,
            'score': score
        })
        if score > best_manual_score:
            best_manual_score = score
            best_manual_params = {
                'n_estimators': n_est,
                'max_depth': max_d, 
                'learning_rate': lr,
                'subsample': subsample,
                'colsample_bytree': colsample,
                'reg_alpha': alpha,
                'reg_lambda': lambda_,
                'min_child_weight': min_child
            }
        print(f"  ✅ Configuración: {n_est} árboles, {max_d} profundidad -> Score: {score:.4f}")
    except Exception as e:
        print(f"  ❌ Error en configuración: {e}")
        continue

print(f"✅ Búsqueda manual completada")
print(f"🏆 Mejores parámetros manuales: {best_manual_params}")
print(f"📈 Mejor score manual: {best_manual_score:.4f}")

# Entrenar modelo manual final
xgb_manual = Pipeline([
    ('preprocessor', preprocessor),
    ('xgb', xgb.XGBClassifier(**best_manual_params, random_state=42, n_jobs=-1, tree_method='hist'))
])
xgb_manual.fit(X_train, y_train_encoded)

# =============================================================================
# COMPARACIÓN RÁPIDA DE MODELOS OPTIMIZADOS
# =============================================================================

print(f"\n🔍 COMPARANDO MODELOS OPTIMIZADOS...")

models = {
    'RandomSearch_XGB': random_search.best_estimator_,
    'ManualSearch_XGB': xgb_manual
}

results_advanced = {}

for model_name, model in models.items():
    print(f"\n🎯 Evaluando {model_name}...")
    
    # Predicciones
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    # Métricas
    accuracy = accuracy_score(y_test_encoded, y_pred)
    precision = precision_score(y_test_encoded, y_pred, average='macro')
    recall = recall_score(y_test_encoded, y_pred, average='macro')
    f1 = f1_score(y_test_encoded, y_pred, average='macro')
    
    # Validación cruzada completa
    cv_scores = cross_val_score(
        model, X_train, y_train_encoded, 
        cv=skf, scoring='f1_macro', n_jobs=-1
    )
    
    # Importancia de características
    feature_importance = pd.DataFrame({
        'feature': feature_names,
        'importance': model.named_steps['xgb'].feature_importances_
    }).sort_values('importance', ascending=False)
    
    results_advanced[model_name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'feature_importance': feature_importance,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f"  ✅ F1-Score: {f1:.4f}")
    print(f"  ✅ CV F1-Score: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Seleccionar mejor modelo
best_model_name = max(results_advanced.items(), key=lambda x: x[1]['f1'])[0]
best_model_advanced = results_advanced[best_model_name]['model']
best_results_advanced = results_advanced[best_model_name]

print(f"\n🏆 MEJOR MODELO AVANZADO: {best_model_name}")
print(f"📊 F1-Score: {best_results_advanced['f1']:.4f}")

# =============================================================================
# INTERPRETABILIDAD - PERMUTATION IMPORTANCE (OPTIMIZADO)
# =============================================================================

print(f"\n🔍 CALCULANDO PERMUTATION IMPORTANCE (OPTIMIZADO)...")

# Preparar datos preprocesados
X_test_processed = preprocessor.transform(X_test)

# Muestrear para mayor velocidad (usar solo 1500 muestras)
sample_size = min(1500, X_test_processed.shape[0])
sample_indices = np.random.choice(X_test_processed.shape[0], sample_size, replace=False)
X_test_sampled = X_test_processed[sample_indices]
y_test_sampled = y_test_encoded[sample_indices]

# Calcular permutation importance con muestreo
perm_start = time.time()
perm_importance = permutation_importance(
    best_model_advanced.named_steps['xgb'],
    X_test_sampled,
    y_test_sampled,
    n_repeats=5,  # REDUCIDO para velocidad
    random_state=42,
    n_jobs=-1,
    scoring='f1_macro'
)
perm_time = time.time() - perm_start

print(f"✅ Permutation Importance calculado en {perm_time:.2f}s")

# Crear DataFrame con resultados
perm_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': perm_importance.importances_mean,
    'importance_std': perm_importance.importances_std
}).sort_values('importance_mean', ascending=False)

print("\n📊 TOP 10 CARACTERÍSTICAS MÁS IMPORTANTES (Permutation):")
print(perm_df.head(10).round(4))

# =============================================================================
# ANÁLISIS DE ESTABILIDAD RÁPIDO
# =============================================================================

print(f"\n📊 ANALIZANDO ESTABILIDAD ENTRE MÉTODOS DE OPTIMIZACIÓN...")

# Calcular importancia promedio entre métodos
all_methods_importance = []
for model_name in results_advanced.keys():
    importance_df = results_advanced[model_name]['feature_importance'].set_index('feature')['importance']
    all_methods_importance.append(importance_df)

stability_df = pd.DataFrame(all_methods_importance).T
stability_df.columns = list(results_advanced.keys())
stability_df['mean_importance'] = stability_df.mean(axis=1)
stability_df['std_importance'] = stability_df.std(axis=1)

print("Características más estables entre métodos de optimización:")
stable_features = stability_df.nsmallest(8, 'std_importance')['mean_importance']
print(stable_features.round(4))

# =============================================================================
# VISUALIZACIONES ESENCIALES
# =============================================================================

plt.figure(figsize=(20, 12))

# Gráfico 1: Comparación de modelos optimizados
plt.subplot(2, 3, 1)
model_names = list(results_advanced.keys())
f1_scores = [results_advanced[model]['f1'] for model in model_names]
colors = ['skyblue', 'lightcoral']

bars = plt.bar(model_names, f1_scores, color=colors, alpha=0.8)
plt.ylabel('F1-Score (Test)')
plt.title('Comparación de Modelos Optimizados\nF1-Score en Test')
plt.xticks(rotation=45)
for bar, score in zip(bars, f1_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{score:.3f}', ha='center', va='bottom')

# Gráfico 2: Permutation Importance vs Gini Importance
plt.subplot(2, 3, 2)
top_perm = perm_df.head(12)
plt.barh(range(len(top_perm)), top_perm['importance_mean'], 
         xerr=top_perm['importance_std'], alpha=0.7, capsize=5)
plt.yticks(range(len(top_perm)), top_perm['feature'])
plt.xlabel('Permutation Importance')
plt.title('Top 12 Características - Permutation Importance')
plt.grid(True, alpha=0.3)

# Gráfico 3: Importancia Gini del mejor modelo
plt.subplot(2, 3, 3)
top_gini = best_results_advanced['feature_importance'].head(12)
plt.barh(range(len(top_gini)), top_gini['importance'], alpha=0.7)
plt.yticks(range(len(top_gini)), top_gini['feature'])
plt.xlabel('Gini Importance')
plt.title('Top 12 Características - Gini Importance')
plt.grid(True, alpha=0.3)

# Gráfico 4: Matriz de confusión del mejor modelo
plt.subplot(2, 3, 4)
y_pred_best = best_results_advanced['y_pred']
cm_best = confusion_matrix(y_test_encoded, y_pred_best)
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Matriz de Confusión\n{best_model_name}')
plt.ylabel('Real')
plt.xlabel('Predicho')

# Gráfico 5: Comparación de métodos de búsqueda
plt.subplot(2, 3, 5)
search_methods = ['Random Search', 'Manual Search']
search_scores = [random_search.best_score_, best_manual_score]
search_times = [random_time, 0]

bars = plt.bar(search_methods, search_scores, alpha=0.7, 
               color=['lightgreen', 'orange'])
plt.ylabel('Mejor F1-Score CV')
plt.title('Comparación de Métodos de Búsqueda')
for bar, score in zip(bars, search_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{score:.3f}', ha='center', va='bottom')
plt.grid(True, alpha=0.3)

# Gráfico 6: Curva de importancia acumulativa
plt.subplot(2, 3, 6)
cumulative_importance = best_results_advanced['feature_importance']['importance'].cumsum()
plt.plot(range(1, len(cumulative_importance) + 1), cumulative_importance, linewidth=2)
plt.axhline(y=0.8, color='r', linestyle='--', alpha=0.7, label='80% importancia')
plt.axhline(y=0.9, color='g', linestyle='--', alpha=0.7, label='90% importancia')
plt.xlabel('Número de Características')
plt.ylabel('Importancia Acumulativa')
plt.title('Importancia Acumulativa de Características')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# =============================================================================
# ANÁLISIS RÁPIDO DE COMPORTAMIENTO
# =============================================================================

print(f"\n📈 ANALIZANDO COMPORTAMIENTO DEL MODELO...")

# Usar resultados de Randomized Search para análisis
results_df = pd.DataFrame(random_search.cv_results_)
train_scores = results_df['mean_train_score']
test_scores = results_df['mean_test_score']
gap = train_scores - test_scores

print(f"📊 Análisis de sobreajuste:")
print(f"  • Diferencia promedio train-test: {gap.mean():.4f}")
print(f"  • Configuraciones probadas: {len(results_df)}")
print(f"  • Mejor configuración: {random_search.best_score_:.4f}")

# =============================================================================
# RESUMEN FINAL OPTIMIZADO
# =============================================================================

end_time = time.time()
execution_time = end_time - start_time

print("\n" + "="*80)
print("🎯 RESUMEN FINAL - XGBOOST AVANZADO OPTIMIZADO")
print("="*80)

print(f"⏱️  Tiempo total ejecución: {execution_time:.2f}s")
print(f"🏆 Mejor modelo: {best_model_name}")
print(f"📈 F1-Score Test: {best_results_advanced['f1']:.4f} ({best_results_advanced['f1']*100:.2f}%)")
print(f"🎯 Validación Cruzada: {best_results_advanced['cv_mean']:.4f} ± {best_results_advanced['cv_std']*2:.4f}")

print(f"\n🔧 MEJORES HIPERPARÁMETROS:")
if best_model_name == 'RandomSearch_XGB':
    best_params = random_search.best_params_
else:
    best_params = best_manual_params

for param, value in best_params.items():
    param_name = param.replace('xgb__', '')
    print(f"  {param_name}: {value}")

print(f"\n📊 CARACTERÍSTICAS MÁS IMPORTANTES:")
for i in range(5):
    feature = perm_df.iloc[i]['feature']
    importance = perm_df.iloc[i]['importance_mean']
    print(f"  {i+1}. {feature}: {importance:.4f}")

print(f"\n⚡ EFICIENCIA COMPUTACIONAL:")
print(f"  Randomized Search: {random_time:.2f}s (25 iteraciones, 5-folds)")
print(f"  Búsqueda Manual: 5 combinaciones (3-folds)")
print(f"  Permutation Importance: {perm_time:.2f}s (5 repeticiones, 1500 muestras)")

print(f"\n🎯 ESTABILIDAD DEL MODELO:")
print(f"  Característica más estable: {stable_features.index[0]}")
print(f"  Número de características con >80% importancia: {(cumulative_importance <= 0.8).sum()}")

print(f"\n📋 REPORTE CLASIFICACIÓN ({best_model_name}):")
print(classification_report(y_test_encoded, y_pred_best, target_names=le.classes_))

# =============================================================================
# GUARDADO DE RESULTADOS
# =============================================================================

advanced_results = {
    'best_model': best_model_advanced,
    'best_model_name': best_model_name,
    'best_params': best_params,
    'random_search_results': random_search.cv_results_,
    'manual_search_results': manual_results,
    'permutation_importance': perm_df,
    'stability_analysis': stability_df,
    'feature_names': feature_names,
    'results_advanced': results_advanced,
    'execution_time': execution_time,
    'preprocessor': preprocessor,
    'label_encoder': le
}

# joblib.dump(advanced_results, 'xgboost_advanced_optimized.pkl')
# print(f"\n💾 Resultados avanzados guardados: 'xgboost_advanced_optimized.pkl'")

# Guardar el mejor modelo por separado
joblib.dump(best_model_advanced, 'best_xgboost_model_optimized.pkl')
print(f"💾 Mejor modelo guardado: 'best_xgboost_model_optimized.pkl'")

print("="*80)
print("✅ ANÁLISIS COMPLETADO - XGBOOST AVANZADO OPTIMIZADO")
print("="*80)


🎯 INICIANDO OPTIMIZACIÓN CON RANDOMIZED SEARCH CV...


NameError: name 'xgb' is not defined

# Restantes

In [ ]:
def print_model_summary(model_name, metrics, best_params, times, cv_scores):
    """
    Muestra un resumen completo del modelo en formato tabla con F1-Score prioritario

    Args:
        model_name: Nombre del modelo
        metrics: Dict con métricas (train_f1, test_f1, train_acc, test_acc, etc.)
        best_params: Dict con mejores hiperparámetros
        times: Dict con tiempos (grid_time, perm_time, total_time)
        cv_scores: Array con scores de cross-validation
    """

    # Calcular overfitting
    overfitting_gap = metrics['train_f1_macro'] - metrics['test_f1_macro']

    print("\n" + "="*100)
    print(f"{'RESUMEN FINAL - ' + model_name.upper():^100}")
    print("="*100)

    # 1. MÉTRICAS PRINCIPALES (F1-Score prioritario)
    print("\n📊 MÉTRICAS DE RENDIMIENTO (F1-Score Prioritario)")
    print("-" * 100)

    # Crear tabla de métricas
    metrics_data = {
        'Métrica': ['F1-Score', 'Accuracy', 'Balanced Accuracy', 'Precision', 'Recall'],
        'Train': [
            f"{metrics['train_f1_macro']:.4f} ({metrics['train_f1_macro']*100:.2f}%)",
            f"{metrics['train_accuracy']:.4f} ({metrics['train_accuracy']*100:.2f}%)",
            f"{metrics['train_balanced_accuracy']:.4f} ({metrics['train_balanced_accuracy']*100:.2f}%)",
            f"{metrics.get('train_precision_macro', 0):.4f}" if 'train_precision_macro' in metrics else "N/A",
            f"{metrics.get('train_recall_macro', 0):.4f}" if 'train_recall_macro' in metrics else "N/A"
        ],
        'Test': [
            f"{metrics['test_f1_macro']:.4f} ({metrics['test_f1_macro']*100:.2f}%)",
            f"{metrics['test_accuracy']:.4f} ({metrics['test_accuracy']*100:.2f}%)",
            f"{metrics['test_balanced_accuracy']:.4f} ({metrics['test_balanced_accuracy']*100:.2f}%)",
            f"{metrics['test_precision_macro']:.4f} ({metrics['test_precision_macro']*100:.2f}%)",
            f"{metrics['test_recall_macro']:.4f} ({metrics['test_recall_macro']*100:.2f}%)"
        ],
        'Diferencia': [
            f"{overfitting_gap:+.4f}",
            f"{metrics['train_accuracy'] - metrics['test_accuracy']:+.4f}",
            f"{metrics['train_balanced_accuracy'] - metrics['test_balanced_accuracy']:+.4f}",
            "-",
            "-"
        ]
    }

    metrics_df = pd.DataFrame(metrics_data)
    print(metrics_df.to_string(index=False))

    # 2. VALIDACIÓN CRUZADA
    print(f"\n\n🎯 VALIDACIÓN CRUZADA (5-Folds)")
    print("-" * 100)
    cv_data = {
        'Métrica': ['F1-Score CV'],
        'Media': [f"{cv_scores.mean():.4f}"],
        'Desv. Est.': [f"{cv_scores.std():.4f}"],
        'Intervalo 95%': [f"{cv_scores.mean():.4f} ± {cv_scores.std()*2:.4f}"],
        'Min': [f"{cv_scores.min():.4f}"],
        'Max': [f"{cv_scores.max():.4f}"]
    }
    cv_df = pd.DataFrame(cv_data)
    print(cv_df.to_string(index=False))

    # 3. ANÁLISIS DE OVERFITTING
    print(f"\n\n⚠️  ANÁLISIS DE OVERFITTING")
    print("-" * 100)
    overfitting_status = "✓ Buen ajuste" if abs(overfitting_gap) < 0.05 else "⚠ Ligero sobreajuste" if abs(overfitting_gap) < 0.10 else "✗ Sobreajuste significativo"

    overfitting_data = {
        'Métrica': ['F1-Score Gap (Train - Test)', 'Estado'],
        'Valor': [f"{overfitting_gap:+.4f} ({abs(overfitting_gap)*100:.2f}%)", overfitting_status]
    }
    overfitting_df = pd.DataFrame(overfitting_data)
    print(overfitting_df.to_string(index=False))

    # 4. MEJORES HIPERPARÁMETROS
    print(f"\n\n⚙️  MEJORES HIPERPARÁMETROS")
    print("-" * 100)
    params_clean = {k.replace('classifier__', ''): v for k, v in best_params.items()}
    params_data = {
        'Parámetro': list(params_clean.keys()),
        'Valor': [str(v) for v in params_clean.values()]
    }
    params_df = pd.DataFrame(params_data)
    print(params_df.to_string(index=False))

    # 5. EFICIENCIA COMPUTACIONAL
    print(f"\n\n⏱️  EFICIENCIA COMPUTACIONAL")
    print("-" * 100)
    time_data = {
        'Etapa': ['Grid/Random Search', 'Permutation Importance', 'Validación Cruzada', 'Total'],
        'Tiempo (s)': [
            f"{times['grid_time']:.2f}",
            f"{times.get('perm_time', 0):.2f}",
            f"{times.get('cv_time', 0):.2f}",
            f"{times['total_time']:.2f}"
        ],
        'Tiempo (min)': [
            f"{times['grid_time']/60:.2f}",
            f"{times.get('perm_time', 0)/60:.2f}",
            f"{times.get('cv_time', 0)/60:.2f}",
            f"{times['total_time']/60:.2f}"
        ]
    }
    time_df = pd.DataFrame(time_data)
    print(time_df.to_string(index=False))

    # 6. RESUMEN EJECUTIVO
    print(f"\n\n📋 RESUMEN EJECUTIVO")
    print("-" * 100)
    print(f"🏆 Modelo: {model_name}")
    print(f"📈 F1-Score Test (Métrica Principal): {metrics['test_f1_macro']:.4f} ({metrics['test_f1_macro']*100:.2f}%)")
    print(f"✓  Validación Cruzada: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
    print(f"⚠  Gap Train-Test: {overfitting_gap:+.4f} ({overfitting_status})")
    print(f"⏱  Tiempo Total: {times['total_time']/60:.2f} min")

    print("\n" + "="*100)
    print(f"{'✓ ANÁLISIS COMPLETADO - ' + model_name.upper():^100}")
    print("="*100 + "\n")


In [ ]:
# ============================================================================
# LOGISTIC REGRESSION - TUNING OPTIMIZADO
# ============================================================================

# ============================================================================
# FUNCIÓN DE EVALUACIÓN Y GUARDADO
# ============================================================================

def evaluate_and_save_model(model, model_name, X_train, X_test, y_train, y_test,
                           best_params, training_time, cv_scores, grid_search):
    """
    Evalua el modelo y guarda resultados en archivo .pkl
    """
    print(f"\n{'='*80}")
    print(f"EVALUANDO MODELO: {model_name}")
    print(f"{'='*80}")
    
    # Predicciones
    print("Generando predicciones...")
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    # Metricas
    train_f1 = f1_score(y_train, y_pred_train, average='macro')
    test_f1 = f1_score(y_test, y_pred_test, average='macro')
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    train_bal_acc = balanced_accuracy_score(y_train, y_pred_train)
    test_bal_acc = balanced_accuracy_score(y_test, y_pred_test)
    
    # Diccionario de resultados
    results = {
        'model': model,
        'model_name': model_name,
        'best_params': best_params,
        'training_time': training_time,
        'cv_scores': {
            'mean': cv_scores.mean(),
            'std': cv_scores.std(),
            'scores': cv_scores
        },
        'metrics': {
            'train_f1_macro': train_f1,
            'test_f1_macro': test_f1,
            'train_accuracy': train_acc,
            'test_accuracy': test_acc,
            'train_balanced_accuracy': train_bal_acc,
            'test_balanced_accuracy': test_bal_acc
        },
        'predictions': {
            'y_train_pred': y_pred_train,
            'y_test_pred': y_pred_test
        },
        'confusion_matrix': confusion_matrix(y_test, y_pred_test),
        'classification_report': classification_report(y_test, y_pred_test),
        'grid_search_results': {
            'cv_results': grid_search.cv_results_,
            'best_index': grid_search.best_index_,
            'best_score': grid_search.best_score_
        }
    }
    
    # Mostrar resultados
    print(f"\nMejores hiperparametros:")
    for param, value in best_params.items():
        print(f"  {param}: {value}")
    
    print(f"\nValidacion Cruzada:")
    print(f"  CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    
    print(f"\nMetricas de Entrenamiento:")
    print(f"  F1-Macro: {train_f1:.4f}")
    print(f"  Accuracy: {train_acc:.4f}")
    print(f"  Balanced Accuracy: {train_bal_acc:.4f}")
    
    print(f"\nMetricas de Prueba:")
    print(f"  F1-Macro: {test_f1:.4f}")
    print(f"  Accuracy: {test_acc:.4f}")
    print(f"  Balanced Accuracy: {test_bal_acc:.4f}")
    
    print(f"\nTiempo de entrenamiento: {training_time:.2f}s ({training_time/60:.2f} min)")
    
    # Guardar resultados
    os.makedirs('tuning_results', exist_ok=True)
    filename = f"tuning_results/{model_name}_results.pkl"
    joblib.dump(results, filename)
    print(f"\nResultados guardados en: {filename}")
    print(f"{'='*80}\n")
    
    return results

print("Funcion evaluate_and_save_model definida correctamente\n")








print("\n" + "="*80)
print("MODELO 1/5: LOGISTIC REGRESSION")
print("="*80)

try:
    from sklearn.model_selection import RandomizedSearchCV
    from scipy.stats import loguniform
    
    # Pipeline
    lr_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            max_iter=1000,  # Reducido de 2000
            random_state=RANDOM_STATE,
            tol=1e-4,  # Tolerancia para convergencia más rápida
            warm_start=False
        ))
    ])
    
    # Grid optimizado - separar por penalty
    lr_param_grid = [
        # L1 penalty
        {
            'classifier__penalty': ['l1'],
            'classifier__C': [0.01, 0.1, 1, 10, 100],
            'classifier__solver': ['saga'],
            'classifier__class_weight': [None, 'balanced']
        },
        # L2 penalty
        {
            'classifier__penalty': ['l2'],
            'classifier__C': [0.01, 0.1, 1, 10, 100],
            'classifier__solver': ['saga'],
            'classifier__class_weight': [None, 'balanced']
        },
        # ElasticNet penalty (el único que usa l1_ratio)
        {
            'classifier__penalty': ['elasticnet'],
            'classifier__C': [0.1, 1, 10],
            'classifier__solver': ['saga'],
            'classifier__l1_ratio': [0.3, 0.5, 0.7],
            'classifier__class_weight': [None, 'balanced']
        }
    ]
    
    # Total combinaciones: (5*2) + (5*2) + (3*3*2) = 10 + 10 + 18 = 38
    print("\nHiperparametros a optimizar:")
    print("   L1: C=[0.01-100], class_weight=[None, balanced]")
    print("   L2: C=[0.01-100], class_weight=[None, balanced]")
    print("   ElasticNet: C=[0.1-10], l1_ratio=[0.3-0.7], class_weight=[None, balanced]")
    print(f"\nCombinaciones totales: 38 (optimizado)\n")
    
    # GridSearch con configuración mejorada
    lr_grid = GridSearchCV(
        lr_pipeline, 
        lr_param_grid, 
        cv=cv_strategy, 
        scoring='f1_macro', 
        n_jobs=-1,  # Usar todos los cores
        verbose=2,
        pre_dispatch='2*n_jobs',  # Mejor manejo de memoria
        error_score='raise'  # Mostrar errores claramente
    )
    
    print("Iniciando busqueda de hiperparametros...\n")
    start = time.time()
    lr_grid.fit(X_train, y_train)
    t_time = time.time() - start
    
    print(f"\nEntrenamiento completado en {t_time:.2f}s ({t_time/60:.2f} min)")
    print(f"Mejor CV score: {lr_grid.best_score_:.4f}")
    print(f"Mejores parametros: {lr_grid.best_params_}")
    
    # CV adicional con el mejor modelo
    print("\nEvaluacion adicional con validacion cruzada...")
    lr_cv = cross_val_score(
        lr_grid.best_estimator_, 
        X_train, y_train, 
        cv=cv_strategy, 
        scoring='f1_macro', 
        n_jobs=-1
    )
    
    # Evaluar y guardar
    lr_results = evaluate_and_save_model(
        model=lr_grid.best_estimator_,
        model_name="Logistic_Regression",
        X_train=X_train, X_test=X_test,
        y_train=y_train, y_test=y_test,
        best_params=lr_grid.best_params_,
        training_time=t_time,
        cv_scores=lr_cv,
        grid_search=lr_grid
    )
    
    print("Logistic Regression - COMPLETADO")
    
except Exception as e:
    print(f"ERROR en Logistic Regression: {e}")
    traceback.print_exc()
    lr_results = None

In [ ]:
# ============================================================================
# RIDGE CLASSIFIER - TUNING
# ============================================================================

print("\n" + "="*80)
print("MODELO: RIDGE CLASSIFIER")
print("="*80)

try:
    from sklearn.linear_model import RidgeClassifier
    from sklearn.inspection import permutation_importance
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score,
                                 precision_score, recall_score,
                                 classification_report, confusion_matrix)

    start_time = time.time()

    # Pipeline
    ridge_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RidgeClassifier(
            random_state=RANDOM_STATE,
            tol=1e-4
        ))
    ])

    # Grid de hiperparámetros
    ridge_param_grid = {
        'classifier__alpha': [0.01, 0.1, 1, 10, 100, 1000],
        'classifier__solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga'],
        'classifier__class_weight': [None, 'balanced']
    }

    print("\nHiperparametros a optimizar:")
    print("   alpha: [0.01, 0.1, 1, 10, 100, 1000]")
    print("   solver: [auto, svd, cholesky, lsqr, sparse_cg, sag, saga]")
    print("   class_weight: [None, balanced]")
    print(f"\nCombinaciones totales: {6 * 7 * 2} = 84\n")

    # GridSearch
    ridge_grid = GridSearchCV(
        ridge_pipeline,
        ridge_param_grid,
        cv=cv_strategy,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=2,
        pre_dispatch='2*n_jobs',
        error_score='raise',
        return_train_score=True
    )

    print("Iniciando busqueda de hiperparametros...\n")
    grid_start = time.time()
    ridge_grid.fit(X_train, y_train)
    grid_time = time.time() - grid_start

    print(f"\nEntrenamiento completado en {grid_time:.2f}s ({grid_time/60:.2f} min)")
    print(f"Mejor CV score (F1-Macro): {ridge_grid.best_score_:.4f}")
    print(f"Mejores parametros: {ridge_grid.best_params_}")

    # =========================================================================
    # ANÁLISIS DE COMPORTAMIENTO
    # =========================================================================

    print(f"\nANALIZANDO COMPORTAMIENTO DEL MODELO...")

    results_df = pd.DataFrame(ridge_grid.cv_results_)
    train_scores = results_df['mean_train_score']
    test_scores = results_df['mean_test_score']
    gap = train_scores - test_scores

    print(f"\nAnalisis de sobreajuste:")
    print(f"  Diferencia promedio train-test: {gap.mean():.4f}")
    print(f"  Configuraciones probadas: {len(results_df)}")

    # =========================================================================
    # OBTENER FEATURE NAMES DEL PIPELINE ENTRENADO
    # =========================================================================

    print(f"\nOBTENIENDO NOMBRES DE FEATURES...")

    fitted_preprocessor = ridge_grid.best_estimator_.named_steps['preprocessor']
    X_train_transformed = fitted_preprocessor.transform(X_train)
    X_test_transformed = fitted_preprocessor.transform(X_test)
    n_features_actual = X_train_transformed.shape[1]

    print(f"Dimensiones despues del preprocesamiento: {X_train_transformed.shape}")
    print(f"Numero real de features: {n_features_actual}")

    try:
        feature_names = fitted_preprocessor.get_feature_names_out()
        print(f"Feature names obtenidos automaticamente: {len(feature_names)}")
    except:
        feature_names = []
        for name, transformer, columns in fitted_preprocessor.transformers_:
            if name == 'num':
                feature_names.extend([f'num__{col}' for col in columns])
            elif name == 'cat':
                if hasattr(transformer.named_steps['onehot'], 'get_feature_names_out'):
                    cat_features = transformer.named_steps['onehot'].get_feature_names_out(columns)
                    feature_names.extend([f'cat__{feat}' for feat in cat_features])
        print(f"Feature names generados manualmente: {len(feature_names)}")

    if len(feature_names) != n_features_actual:
        print(f"ADVERTENCIA: Ajustando feature_names de {len(feature_names)} a {n_features_actual}")
        if len(feature_names) < n_features_actual:
            feature_names = list(feature_names) + [f'feature_{i}' for i in range(len(feature_names), n_features_actual)]
        else:
            feature_names = list(feature_names)[:n_features_actual]

    print(f"Total de features despues del ajuste: {len(feature_names)}")

    # =========================================================================
    # PERMUTATION IMPORTANCE
    # =========================================================================

    print(f"\nCALCULANDO IMPORTANCIA DE FEATURES...")
    perm_start = time.time()

    perm_result = permutation_importance(
        ridge_grid.best_estimator_,
        X_test,
        y_test,
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scoring='f1_macro'
    )

    perm_time = time.time() - perm_start

    print(f"Dimensiones de importancias: {len(perm_result.importances_mean)}")
    print(f"Dimensiones de feature_names: {len(feature_names)}")

    if len(feature_names) != len(perm_result.importances_mean):
        print(f"ADVERTENCIA: Ajustando feature_names de {len(feature_names)} a {len(perm_result.importances_mean)}")
        if len(feature_names) < len(perm_result.importances_mean):
            feature_names = list(feature_names) + [f'feature_{i}' for i in range(len(feature_names), len(perm_result.importances_mean))]
        else:
            feature_names = list(feature_names)[:len(perm_result.importances_mean)]

    perm_df = pd.DataFrame({
        'feature': feature_names,
        'importance_mean': perm_result.importances_mean,
        'importance_std': perm_result.importances_std
    }).sort_values('importance_mean', ascending=False)

    print(f"Permutation Importance calculado en {perm_time:.2f}s")

    # =========================================================================
    # VALIDACIÓN CRUZADA ADICIONAL
    # =========================================================================

    print("\nEvaluacion adicional con validacion cruzada...")
    cv_start = time.time()
    ridge_cv = cross_val_score(
        ridge_grid.best_estimator_,
        X_train, y_train,
        cv=cv_strategy,
        scoring='f1_macro',
        n_jobs=-1
    )
    cv_time = time.time() - cv_start

    # =========================================================================
    # PREDICCIONES Y MÉTRICAS
    # =========================================================================

    print("Generando predicciones...")
    y_pred_train = ridge_grid.best_estimator_.predict(X_train)
    y_pred_test = ridge_grid.best_estimator_.predict(X_test)

    # Métricas completas
    train_f1 = f1_score(y_train, y_pred_train, average='macro')
    test_f1 = f1_score(y_test, y_pred_test, average='macro')
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    train_bal_acc = balanced_accuracy_score(y_train, y_pred_train)
    test_bal_acc = balanced_accuracy_score(y_test, y_pred_test)
    train_precision = precision_score(y_train, y_pred_train, average='macro')
    test_precision = precision_score(y_test, y_pred_test, average='macro')
    train_recall = recall_score(y_train, y_pred_train, average='macro')
    test_recall = recall_score(y_test, y_pred_test, average='macro')

    # =========================================================================
    # MOSTRAR RESUMEN CON FORMATO ESTANDARIZADO
    # =========================================================================

    end_time = time.time()
    execution_time = end_time - start_time

    # Preparar datos para print_model_summary
    metrics = {
        'train_f1_macro': train_f1,
        'test_f1_macro': test_f1,
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'train_balanced_accuracy': train_bal_acc,
        'test_balanced_accuracy': test_bal_acc,
        'train_precision_macro': train_precision,
        'test_precision_macro': test_precision,
        'train_recall_macro': train_recall,
        'test_recall_macro': test_recall
    }

    times = {
        'grid_time': grid_time,
        'perm_time': perm_time,
        'cv_time': cv_time,
        'total_time': execution_time
    }

    # Llamar a la función de resumen
    print_model_summary('Ridge Classifier', metrics, ridge_grid.best_params_, times, ridge_cv)

    # =========================================================================
    # TOP 10 FEATURES MÁS IMPORTANTES
    # =========================================================================

    print("\n📊 TOP 10 CARACTERÍSTICAS MÁS IMPORTANTES")
    print("-" * 100)
    top10_data = {
        'Rank': range(1, 11),
        'Feature': perm_df.head(10)['feature'].values,
        'Importancia': perm_df.head(10)['importance_mean'].values,
        'Desv. Est.': perm_df.head(10)['importance_std'].values
    }
    top10_df = pd.DataFrame(top10_data)
    print(top10_df.to_string(index=False))

    print(f"\n\n📊 REPORTE DE CLASIFICACIÓN DETALLADO")
    print("-" * 100)
    print(classification_report(y_test, y_pred_test))

    # =========================================================================
    # GRÁFICAS
    # =========================================================================

    print("\nGenerando graficas...")

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Ridge Classifier - Análisis Completo', fontsize=16, fontweight='bold')

    # 1. Top 15 Features más importantes
    top_features = perm_df.head(15)
    axes[0, 0].barh(range(len(top_features)), top_features['importance_mean'], color='crimson')
    axes[0, 0].set_yticks(range(len(top_features)))
    axes[0, 0].set_yticklabels(top_features['feature'], fontsize=8)
    axes[0, 0].set_xlabel('Importancia Media (F1-Score)')
    axes[0, 0].set_title('Top 15 Features - Permutation Importance')
    axes[0, 0].invert_yaxis()
    axes[0, 0].grid(True, alpha=0.3, axis='x')

    # 2. Matriz de Confusión
    cm = confusion_matrix(y_test, y_pred_test)
    classes = sorted(y_test.unique())
    sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=axes[0, 1],
                xticklabels=classes, yticklabels=classes)
    axes[0, 1].set_title('Matriz de Confusion')
    axes[0, 1].set_ylabel('Real')
    axes[0, 1].set_xlabel('Prediccion')

    # 3. Train vs Test Scores (Overfitting Analysis)
    axes[1, 0].scatter(train_scores, test_scores, alpha=0.5, color='indianred')
    axes[1, 0].plot([train_scores.min(), train_scores.max()],
                    [train_scores.min(), train_scores.max()],
                    'r--', lw=2, label='Linea perfecta')
    axes[1, 0].set_xlabel('Train F1-Score')
    axes[1, 0].set_ylabel('Test F1-Score')
    axes[1, 0].set_title('Analisis de Sobreajuste')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # 4. Distribución de scores
    axes[1, 1].hist(test_scores, bins=20, alpha=0.7, label='Test F1-Scores',
                    edgecolor='black', color='crimson')
    axes[1, 1].axvline(ridge_grid.best_score_, color='darkred', linestyle='--',
                       linewidth=2, label=f'Best: {ridge_grid.best_score_:.4f}')
    axes[1, 1].set_xlabel('F1-Score')
    axes[1, 1].set_ylabel('Frecuencia')
    axes[1, 1].set_title('Distribucion de F1-Scores en GridSearch')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()

    # Guardar gráficas
    os.makedirs('tuning_results', exist_ok=True)
    plt.savefig('tuning_results/Ridge_Classifier_analysis.png', dpi=300, bbox_inches='tight')
    print("\n✓ Graficas guardadas en: tuning_results/Ridge_Classifier_analysis.png")
    plt.show()

    # =========================================================================
    # GUARDADO DE RESULTADOS
    # =========================================================================

    ridge_results = {
        'model': ridge_grid.best_estimator_,
        'model_name': 'Ridge_Classifier',
        'best_params': ridge_grid.best_params_,
        'training_time': grid_time,
        'total_time': execution_time,
        'grid_search_results': ridge_grid.cv_results_,
        'permutation_importance': perm_df,
        'feature_names': list(feature_names),
        'cv_scores': {
            'mean': ridge_cv.mean(),
            'std': ridge_cv.std(),
            'scores': ridge_cv
        },
        'metrics': {
            'train_f1_macro': train_f1,
            'test_f1_macro': test_f1,
            'train_accuracy': train_acc,
            'test_accuracy': test_acc,
            'train_balanced_accuracy': train_bal_acc,
            'test_balanced_accuracy': test_bal_acc,
            'train_precision_macro': train_precision,
            'test_precision_macro': test_precision,
            'train_recall_macro': train_recall,
            'test_recall_macro': test_recall
        },
        'predictions': {
            'y_train_pred': y_pred_train,
            'y_test_pred': y_pred_test
        },
        'confusion_matrix': cm,
        'classification_report': classification_report(y_test, y_pred_test),
        'overfitting_analysis': {
            'train_test_gap_mean': gap.mean(),
            'train_test_gap_std': gap.std()
        }
    }

    # joblib.dump(ridge_results, 'tuning_results/Ridge_Classifier_results.pkl')
    # print("✓ Resultados completos guardados: tuning_results/Ridge_Classifier_results.pkl")

    joblib.dump(ridge_grid.best_estimator_, 'tuning_results/best_Ridge_Classifier_model.pkl')
    print("✓ Mejor modelo guardado: tuning_results/best_Ridge_Classifier_model.pkl")

except Exception as e:
    print(f"\nERROR en Ridge Classifier: {e}")
    traceback.print_exc()
    ridge_results = None


In [ ]:
# ============================================================================
# LOGISTIC REGRESSION - TUNING
# ============================================================================

print("\n" + "="*80)
print("MODELO 1/5: LOGISTIC REGRESSION")
print("="*80)

try:
    from sklearn.model_selection import RandomizedSearchCV
    from sklearn.inspection import permutation_importance
    from scipy.stats import loguniform
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score,
                                 precision_score, recall_score,
                                 classification_report, confusion_matrix)

    start_time = time.time()

    # Pipeline
    lr_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE,
            tol=1e-4,
            warm_start=False
        ))
    ])

    # Grid optimizado - separar por penalty
    lr_param_grid = [
        # L1 penalty
        {
            'classifier__penalty': ['l1'],
            'classifier__C': [0.01, 0.1, 1, 10, 100],
            'classifier__solver': ['saga'],
            'classifier__class_weight': [None, 'balanced']
        },
        # L2 penalty
        {
            'classifier__penalty': ['l2'],
            'classifier__C': [0.01, 0.1, 1, 10, 100],
            'classifier__solver': ['saga'],
            'classifier__class_weight': [None, 'balanced']
        },
        # ElasticNet penalty
        {
            'classifier__penalty': ['elasticnet'],
            'classifier__C': [0.1, 1, 10],
            'classifier__solver': ['saga'],
            'classifier__l1_ratio': [0.3, 0.5, 0.7],
            'classifier__class_weight': [None, 'balanced']
        }
    ]

    print("\nHiperparametros a optimizar:")
    print("   L1: C=[0.01-100], class_weight=[None, balanced]")
    print("   L2: C=[0.01-100], class_weight=[None, balanced]")
    print("   ElasticNet: C=[0.1-10], l1_ratio=[0.3-0.7], class_weight=[None, balanced]")
    print(f"\nCombinaciones totales: 38 (optimizado)\n")

    # GridSearch
    lr_grid = GridSearchCV(
        lr_pipeline,
        lr_param_grid,
        cv=cv_strategy,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=2,
        pre_dispatch='2*n_jobs',
        error_score='raise',
        return_train_score=True
    )

    print("Iniciando busqueda de hiperparametros...\n")
    grid_start = time.time()
    lr_grid.fit(X_train, y_train)
    grid_time = time.time() - grid_start

    print(f"\nEntrenamiento completado en {grid_time:.2f}s ({grid_time/60:.2f} min)")
    print(f"Mejor CV score (F1-Macro): {lr_grid.best_score_:.4f}")
    print(f"Mejores parametros: {lr_grid.best_params_}")

    # =========================================================================
    # ANÁLISIS DE COMPORTAMIENTO
    # =========================================================================

    print(f"\nANALIZANDO COMPORTAMIENTO DEL MODELO...")

    results_df = pd.DataFrame(lr_grid.cv_results_)
    train_scores = results_df['mean_train_score']
    test_scores = results_df['mean_test_score']
    gap = train_scores - test_scores

    print(f"\nAnalisis de sobreajuste:")
    print(f"  Diferencia promedio train-test: {gap.mean():.4f}")
    print(f"  Configuraciones probadas: {len(results_df)}")

    # =========================================================================
    # OBTENER FEATURE NAMES DEL PIPELINE ENTRENADO
    # =========================================================================

    print(f"\nOBTENIENDO NOMBRES DE FEATURES...")

    fitted_preprocessor = lr_grid.best_estimator_.named_steps['preprocessor']
    X_train_transformed = fitted_preprocessor.transform(X_train)
    X_test_transformed = fitted_preprocessor.transform(X_test)
    n_features_actual = X_train_transformed.shape[1]

    print(f"Dimensiones despues del preprocesamiento: {X_train_transformed.shape}")
    print(f"Numero real de features: {n_features_actual}")

    try:
        feature_names = fitted_preprocessor.get_feature_names_out()
        print(f"Feature names obtenidos automaticamente: {len(feature_names)}")
    except:
        feature_names = []
        for name, transformer, columns in fitted_preprocessor.transformers_:
            if name == 'num':
                feature_names.extend([f'num__{col}' for col in columns])
            elif name == 'cat':
                if hasattr(transformer.named_steps['onehot'], 'get_feature_names_out'):
                    cat_features = transformer.named_steps['onehot'].get_feature_names_out(columns)
                    feature_names.extend([f'cat__{feat}' for feat in cat_features])
        print(f"Feature names generados manualmente: {len(feature_names)}")

    if len(feature_names) != n_features_actual:
        print(f"ADVERTENCIA: Ajustando feature_names de {len(feature_names)} a {n_features_actual}")
        if len(feature_names) < n_features_actual:
            feature_names = list(feature_names) + [f'feature_{i}' for i in range(len(feature_names), n_features_actual)]
        else:
            feature_names = list(feature_names)[:n_features_actual]

    print(f"Total de features despues del ajuste: {len(feature_names)}")

    # =========================================================================
    # PERMUTATION IMPORTANCE
    # =========================================================================

    print(f"\nCALCULANDO IMPORTANCIA DE FEATURES...")
    perm_start = time.time()

    perm_result = permutation_importance(
        lr_grid.best_estimator_,
        X_test,
        y_test,
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scoring='f1_macro'
    )

    perm_time = time.time() - perm_start

    print(f"Dimensiones de importancias: {len(perm_result.importances_mean)}")
    print(f"Dimensiones de feature_names: {len(feature_names)}")

    if len(feature_names) != len(perm_result.importances_mean):
        print(f"ADVERTENCIA: Ajustando feature_names de {len(feature_names)} a {len(perm_result.importances_mean)}")
        if len(feature_names) < len(perm_result.importances_mean):
            feature_names = list(feature_names) + [f'feature_{i}' for i in range(len(feature_names), len(perm_result.importances_mean))]
        else:
            feature_names = list(feature_names)[:len(perm_result.importances_mean)]

    perm_df = pd.DataFrame({
        'feature': feature_names,
        'importance_mean': perm_result.importances_mean,
        'importance_std': perm_result.importances_std
    }).sort_values('importance_mean', ascending=False)

    print(f"Permutation Importance calculado en {perm_time:.2f}s")

    # =========================================================================
    # VALIDACIÓN CRUZADA ADICIONAL
    # =========================================================================

    print("\nEvaluacion adicional con validacion cruzada...")
    cv_start = time.time()
    lr_cv = cross_val_score(
        lr_grid.best_estimator_,
        X_train, y_train,
        cv=cv_strategy,
        scoring='f1_macro',
        n_jobs=-1
    )
    cv_time = time.time() - cv_start

    # =========================================================================
    # PREDICCIONES Y MÉTRICAS
    # =========================================================================

    print("Generando predicciones...")
    y_pred_train = lr_grid.best_estimator_.predict(X_train)
    y_pred_test = lr_grid.best_estimator_.predict(X_test)

    # Métricas completas
    train_f1 = f1_score(y_train, y_pred_train, average='macro')
    test_f1 = f1_score(y_test, y_pred_test, average='macro')
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    train_bal_acc = balanced_accuracy_score(y_train, y_pred_train)
    test_bal_acc = balanced_accuracy_score(y_test, y_pred_test)
    train_precision = precision_score(y_train, y_pred_train, average='macro')
    test_precision = precision_score(y_test, y_pred_test, average='macro')
    train_recall = recall_score(y_train, y_pred_train, average='macro')
    test_recall = recall_score(y_test, y_pred_test, average='macro')

    # =========================================================================
    # MOSTRAR RESUMEN CON FORMATO ESTANDARIZADO
    # =========================================================================

    end_time = time.time()
    execution_time = end_time - start_time

    # Preparar datos para print_model_summary
    metrics = {
        'train_f1_macro': train_f1,
        'test_f1_macro': test_f1,
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'train_balanced_accuracy': train_bal_acc,
        'test_balanced_accuracy': test_bal_acc,
        'train_precision_macro': train_precision,
        'test_precision_macro': test_precision,
        'train_recall_macro': train_recall,
        'test_recall_macro': test_recall
    }

    times = {
        'grid_time': grid_time,
        'perm_time': perm_time,
        'cv_time': cv_time,
        'total_time': execution_time
    }

    # Llamar a la función de resumen
    print_model_summary('Logistic Regression', metrics, lr_grid.best_params_, times, lr_cv)

    # =========================================================================
    # TOP 10 FEATURES MÁS IMPORTANTES
    # =========================================================================

    print("\n📊 TOP 10 CARACTERÍSTICAS MÁS IMPORTANTES")
    print("-" * 100)
    top10_data = {
        'Rank': range(1, 11),
        'Feature': perm_df.head(10)['feature'].values,
        'Importancia': perm_df.head(10)['importance_mean'].values,
        'Desv. Est.': perm_df.head(10)['importance_std'].values
    }
    top10_df = pd.DataFrame(top10_data)
    print(top10_df.to_string(index=False))

    print(f"\n\n📊 REPORTE DE CLASIFICACIÓN DETALLADO")
    print("-" * 100)
    print(classification_report(y_test, y_pred_test))

    # =========================================================================
    # GRÁFICAS
    # =========================================================================

    print("\nGenerando graficas...")

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Logistic Regression - Análisis Completo', fontsize=16, fontweight='bold')

    # 1. Top 15 Features más importantes
    top_features = perm_df.head(15)
    axes[0, 0].barh(range(len(top_features)), top_features['importance_mean'])
    axes[0, 0].set_yticks(range(len(top_features)))
    axes[0, 0].set_yticklabels(top_features['feature'], fontsize=8)
    axes[0, 0].set_xlabel('Importancia Media (F1-Score)')
    axes[0, 0].set_title('Top 15 Features - Permutation Importance')
    axes[0, 0].invert_yaxis()
    axes[0, 0].grid(True, alpha=0.3, axis='x')

    # 2. Matriz de Confusión
    cm = confusion_matrix(y_test, y_pred_test)
    classes = sorted(y_test.unique())
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1],
                xticklabels=classes, yticklabels=classes)
    axes[0, 1].set_title('Matriz de Confusion')
    axes[0, 1].set_ylabel('Real')
    axes[0, 1].set_xlabel('Prediccion')

    # 3. Train vs Test Scores (Overfitting Analysis)
    axes[1, 0].scatter(train_scores, test_scores, alpha=0.5)
    axes[1, 0].plot([train_scores.min(), train_scores.max()],
                    [train_scores.min(), train_scores.max()],
                    'r--', lw=2, label='Linea perfecta')
    axes[1, 0].set_xlabel('Train F1-Score')
    axes[1, 0].set_ylabel('Test F1-Score')
    axes[1, 0].set_title('Analisis de Sobreajuste')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # 4. Distribución de scores
    axes[1, 1].hist(test_scores, bins=20, alpha=0.7, label='Test F1-Scores',
                    edgecolor='black', color='steelblue')
    axes[1, 1].axvline(lr_grid.best_score_, color='r', linestyle='--',
                       linewidth=2, label=f'Best: {lr_grid.best_score_:.4f}')
    axes[1, 1].set_xlabel('F1-Score')
    axes[1, 1].set_ylabel('Frecuencia')
    axes[1, 1].set_title('Distribucion de F1-Scores en GridSearch')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()

    # Guardar gráficas
    os.makedirs('tuning_results', exist_ok=True)
    plt.savefig('tuning_results/Logistic_Regression_analysis.png', dpi=300, bbox_inches='tight')
    print("\n✓ Graficas guardadas en: tuning_results/Logistic_Regression_analysis.png")
    plt.show()

    # =========================================================================
    # GUARDADO DE RESULTADOS
    # =========================================================================

    lr_results = {
        'model': lr_grid.best_estimator_,
        'model_name': 'Logistic_Regression',
        'best_params': lr_grid.best_params_,
        'training_time': grid_time,
        'total_time': execution_time,
        'grid_search_results': lr_grid.cv_results_,
        'permutation_importance': perm_df,
        'feature_names': list(feature_names),
        'cv_scores': {
            'mean': lr_cv.mean(),
            'std': lr_cv.std(),
            'scores': lr_cv
        },
        'metrics': {
            'train_f1_macro': train_f1,
            'test_f1_macro': test_f1,
            'train_accuracy': train_acc,
            'test_accuracy': test_acc,
            'train_balanced_accuracy': train_bal_acc,
            'test_balanced_accuracy': test_bal_acc,
            'train_precision_macro': train_precision,
            'test_precision_macro': test_precision,
            'train_recall_macro': train_recall,
            'test_recall_macro': test_recall
        },
        'predictions': {
            'y_train_pred': y_pred_train,
            'y_test_pred': y_pred_test
        },
        'confusion_matrix': cm,
        'classification_report': classification_report(y_test, y_pred_test),
        'overfitting_analysis': {
            'train_test_gap_mean': gap.mean(),
            'train_test_gap_std': gap.std()
        }
    }

    # joblib.dump(lr_results, 'tuning_results/Logistic_Regression_results.pkl')
    # print("✓ Resultados completos guardados: tuning_results/Logistic_Regression_results.pkl")

    joblib.dump(lr_grid.best_estimator_, 'tuning_results/best_Logistic_Regression_model.pkl')
    print("✓ Mejor modelo guardado: tuning_results/best_Logistic_Regression_model.pkl")

except Exception as e:
    print(f"\nERROR en Logistic Regression: {e}")
    traceback.print_exc()
    lr_results = None


In [ ]:
# ============================================================================
# DECISION TREE - TUNING
# ============================================================================

print("\n" + "="*80)
print("MODELO: DECISION TREE")
print("="*80)

try:
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.model_selection import RandomizedSearchCV
    from sklearn.inspection import permutation_importance
    from scipy.stats import randint, uniform
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score,
                                 precision_score, recall_score,
                                 classification_report, confusion_matrix)
    import numpy as np

    start_time = time.time()

    # Pipeline
    dt_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', DecisionTreeClassifier(
            random_state=RANDOM_STATE
        ))
    ])

    # Distribuciones de hiperparámetros para RandomizedSearchCV
    dt_param_distributions = {
        'classifier__criterion': ['gini', 'entropy', 'log_loss'],
        'classifier__max_depth': [None, 5, 10, 15, 20, 25, 30, 40, 50],
        'classifier__min_samples_split': randint(2, 50),
        'classifier__min_samples_leaf': randint(1, 30),
        'classifier__max_features': [None, 'sqrt', 'log2', 0.5, 0.7, 0.9],
        'classifier__class_weight': [None, 'balanced'],
        'classifier__splitter': ['best', 'random'],
        'classifier__min_impurity_decrease': uniform(0.0, 0.01)
    }

    print("\nHiperparametros a optimizar:")
    print("   criterion: [gini, entropy, log_loss]")
    print("   max_depth: [None, 5, 10, 15, 20, 25, 30, 40, 50]")
    print("   min_samples_split: randint(2, 50)")
    print("   min_samples_leaf: randint(1, 30)")
    print("   max_features: [None, sqrt, log2, 0.5, 0.7, 0.9]")
    print("   class_weight: [None, balanced]")
    print("   splitter: [best, random]")
    print("   min_impurity_decrease: uniform(0.0, 0.01)")
    print(f"\nIteraciones de RandomizedSearchCV: 100\n")

    # RandomizedSearchCV
    dt_grid = RandomizedSearchCV(
        dt_pipeline,
        dt_param_distributions,
        n_iter=100,
        cv=cv_strategy,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=2,
        random_state=RANDOM_STATE,
        pre_dispatch='2*n_jobs',
        error_score='raise',
        return_train_score=True
    )

    print("Iniciando busqueda de hiperparametros...\n")
    grid_start = time.time()
    dt_grid.fit(X_train, y_train)
    grid_time = time.time() - grid_start

    print(f"\nEntrenamiento completado en {grid_time:.2f}s ({grid_time/60:.2f} min)")
    print(f"Mejor CV score (F1-Macro): {dt_grid.best_score_:.4f}")
    print(f"Mejores parametros: {dt_grid.best_params_}")

    # =========================================================================
    # ANÁLISIS DE COMPORTAMIENTO
    # =========================================================================

    print(f"\nANALIZANDO COMPORTAMIENTO DEL MODELO...")

    results_df = pd.DataFrame(dt_grid.cv_results_)
    train_scores = results_df['mean_train_score']
    test_scores = results_df['mean_test_score']
    gap = train_scores - test_scores

    print(f"\nAnalisis de sobreajuste:")
    print(f"  Diferencia promedio train-test: {gap.mean():.4f}")
    print(f"  Configuraciones probadas: {len(results_df)}")

    # =========================================================================
    # OBTENER FEATURE NAMES DEL PIPELINE ENTRENADO
    # =========================================================================

    print(f"\nOBTENIENDO NOMBRES DE FEATURES...")

    fitted_preprocessor = dt_grid.best_estimator_.named_steps['preprocessor']
    X_train_transformed = fitted_preprocessor.transform(X_train)
    X_test_transformed = fitted_preprocessor.transform(X_test)
    n_features_actual = X_train_transformed.shape[1]

    print(f"Dimensiones despues del preprocesamiento: {X_train_transformed.shape}")
    print(f"Numero real de features: {n_features_actual}")

    try:
        feature_names = fitted_preprocessor.get_feature_names_out()
        print(f"Feature names obtenidos automaticamente: {len(feature_names)}")
    except:
        feature_names = []
        for name, transformer, columns in fitted_preprocessor.transformers_:
            if name == 'num':
                feature_names.extend([f'num__{col}' for col in columns])
            elif name == 'cat':
                if hasattr(transformer.named_steps['onehot'], 'get_feature_names_out'):
                    cat_features = transformer.named_steps['onehot'].get_feature_names_out(columns)
                    feature_names.extend([f'cat__{feat}' for feat in cat_features])
        print(f"Feature names generados manualmente: {len(feature_names)}")

    if len(feature_names) != n_features_actual:
        print(f"ADVERTENCIA: Ajustando feature_names de {len(feature_names)} a {n_features_actual}")
        if len(feature_names) < n_features_actual:
            feature_names = list(feature_names) + [f'feature_{i}' for i in range(len(feature_names), n_features_actual)]
        else:
            feature_names = list(feature_names)[:n_features_actual]

    print(f"Total de features despues del ajuste: {len(feature_names)}")

    # =========================================================================
    # NATIVE FEATURE IMPORTANCE (Decision Tree)
    # =========================================================================

    print(f"\nOBTENIENDO IMPORTANCIA NATIVA DE FEATURES (Decision Tree)...")
    best_dt_classifier = dt_grid.best_estimator_.named_steps['classifier']
    native_importances = best_dt_classifier.feature_importances_

    native_df = pd.DataFrame({
        'feature': feature_names,
        'importance': native_importances
    }).sort_values('importance', ascending=False)

    print(f"Native Feature Importance calculado")

    # =========================================================================
    # PERMUTATION IMPORTANCE
    # =========================================================================

    print(f"\nCALCULANDO IMPORTANCIA DE FEATURES (Permutation)...")
    perm_start = time.time()

    perm_result = permutation_importance(
        dt_grid.best_estimator_,
        X_test,
        y_test,
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scoring='f1_macro'
    )

    perm_time = time.time() - perm_start

    print(f"Dimensiones de importancias: {len(perm_result.importances_mean)}")
    print(f"Dimensiones de feature_names: {len(feature_names)}")

    if len(feature_names) != len(perm_result.importances_mean):
        print(f"ADVERTENCIA: Ajustando feature_names de {len(feature_names)} a {len(perm_result.importances_mean)}")
        if len(feature_names) < len(perm_result.importances_mean):
            feature_names = list(feature_names) + [f'feature_{i}' for i in range(len(feature_names), len(perm_result.importances_mean))]
        else:
            feature_names = list(feature_names)[:len(perm_result.importances_mean)]

    perm_df = pd.DataFrame({
        'feature': feature_names,
        'importance_mean': perm_result.importances_mean,
        'importance_std': perm_result.importances_std
    }).sort_values('importance_mean', ascending=False)

    print(f"Permutation Importance calculado en {perm_time:.2f}s")

    # =========================================================================
    # DEPTH VS PERFORMANCE ANALYSIS
    # =========================================================================

    print(f"\nANALIZANDO PROFUNDIDAD VS PERFORMANCE...")
    depth_analysis = results_df[['param_classifier__max_depth', 'mean_test_score', 'mean_train_score']].copy()
    depth_analysis['param_classifier__max_depth'] = depth_analysis['param_classifier__max_depth'].fillna('None')
    depth_grouped = depth_analysis.groupby('param_classifier__max_depth').agg({
        'mean_test_score': 'mean',
        'mean_train_score': 'mean'
    }).reset_index()

    # =========================================================================
    # VALIDACIÓN CRUZADA ADICIONAL
    # =========================================================================

    print("\nEvaluacion adicional con validacion cruzada...")
    cv_start = time.time()
    dt_cv = cross_val_score(
        dt_grid.best_estimator_,
        X_train, y_train,
        cv=cv_strategy,
        scoring='f1_macro',
        n_jobs=-1
    )
    cv_time = time.time() - cv_start

    # =========================================================================
    # PREDICCIONES Y MÉTRICAS
    # =========================================================================

    print("Generando predicciones...")
    y_pred_train = dt_grid.best_estimator_.predict(X_train)
    y_pred_test = dt_grid.best_estimator_.predict(X_test)

    # Métricas completas
    train_f1 = f1_score(y_train, y_pred_train, average='macro')
    test_f1 = f1_score(y_test, y_pred_test, average='macro')
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    train_bal_acc = balanced_accuracy_score(y_train, y_pred_train)
    test_bal_acc = balanced_accuracy_score(y_test, y_pred_test)
    train_precision = precision_score(y_train, y_pred_train, average='macro')
    test_precision = precision_score(y_test, y_pred_test, average='macro')
    train_recall = recall_score(y_train, y_pred_train, average='macro')
    test_recall = recall_score(y_test, y_pred_test, average='macro')

    # =========================================================================
    # MOSTRAR RESUMEN CON FORMATO ESTANDARIZADO
    # =========================================================================

    end_time = time.time()
    execution_time = end_time - start_time

    # Preparar datos para print_model_summary
    metrics = {
        'train_f1_macro': train_f1,
        'test_f1_macro': test_f1,
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'train_balanced_accuracy': train_bal_acc,
        'test_balanced_accuracy': test_bal_acc,
        'train_precision_macro': train_precision,
        'test_precision_macro': test_precision,
        'train_recall_macro': train_recall,
        'test_recall_macro': test_recall
    }

    times = {
        'grid_time': grid_time,
        'perm_time': perm_time,
        'cv_time': cv_time,
        'total_time': execution_time
    }

    # Llamar a la función de resumen
    print_model_summary('Decision Tree', metrics, dt_grid.best_params_, times, dt_cv)

    # =========================================================================
    # TOP 10 FEATURES MÁS IMPORTANTES
    # =========================================================================

    print("\n📊 TOP 10 CARACTERÍSTICAS MÁS IMPORTANTES")
    print("-" * 100)
    top10_data = {
        'Rank': range(1, 11),
        'Feature': perm_df.head(10)['feature'].values,
        'Importancia': perm_df.head(10)['importance_mean'].values,
        'Desv. Est.': perm_df.head(10)['importance_std'].values
    }
    top10_df = pd.DataFrame(top10_data)
    print(top10_df.to_string(index=False))

    print(f"\n\n📊 REPORTE DE CLASIFICACIÓN DETALLADO")
    print("-" * 100)
    print(classification_report(y_test, y_pred_test))

    # =========================================================================
    # GRÁFICAS (3x2 layout)
    # =========================================================================

    print("\nGenerando graficas...")

    fig, axes = plt.subplots(3, 2, figsize=(15, 18))
    fig.suptitle('Decision Tree - Análisis Completo', fontsize=16, fontweight='bold')

    # 1. Top 15 Features - Permutation Importance
    top_features = perm_df.head(15)
    axes[0, 0].barh(range(len(top_features)), top_features['importance_mean'], color='forestgreen')
    axes[0, 0].set_yticks(range(len(top_features)))
    axes[0, 0].set_yticklabels(top_features['feature'], fontsize=8)
    axes[0, 0].set_xlabel('Importancia Media (F1-Score)')
    axes[0, 0].set_title('Top 15 Features - Permutation Importance')
    axes[0, 0].invert_yaxis()
    axes[0, 0].grid(True, alpha=0.3, axis='x')

    # 2. Native Feature Importance (Decision Tree)
    top_native = native_df.head(15)
    axes[0, 1].barh(range(len(top_native)), top_native['importance'], color='darkorange')
    axes[0, 1].set_yticks(range(len(top_native)))
    axes[0, 1].set_yticklabels(top_native['feature'], fontsize=8)
    axes[0, 1].set_xlabel('Importancia (Gini/Entropy)')
    axes[0, 1].set_title('Top 15 Features - Native Feature Importance')
    axes[0, 1].invert_yaxis()
    axes[0, 1].grid(True, alpha=0.3, axis='x')

    # 3. Matriz de Confusión
    cm = confusion_matrix(y_test, y_pred_test)
    classes = sorted(y_test.unique())
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[1, 0],
                xticklabels=classes, yticklabels=classes)
    axes[1, 0].set_title('Matriz de Confusion')
    axes[1, 0].set_ylabel('Real')
    axes[1, 0].set_xlabel('Prediccion')

    # 4. Depth vs Performance
    # Convertir 'None' a un valor numérico para graficar
    depth_plot = depth_grouped.copy()
    depth_plot['depth_numeric'] = depth_plot['param_classifier__max_depth'].apply(
        lambda x: 100 if x == 'None' else int(x)
    )
    depth_plot = depth_plot.sort_values('depth_numeric')

    axes[1, 1].plot(depth_plot['depth_numeric'], depth_plot['mean_train_score'],
                    marker='o', label='Train F1-Score', color='forestgreen', linewidth=2)
    axes[1, 1].plot(depth_plot['depth_numeric'], depth_plot['mean_test_score'],
                    marker='s', label='Test F1-Score', color='darkorange', linewidth=2)
    axes[1, 1].set_xlabel('Max Depth (100 = None)')
    axes[1, 1].set_ylabel('F1-Score')
    axes[1, 1].set_title('Profundidad vs Performance')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    # 5. Train vs Test Scores (Overfitting Analysis)
    axes[2, 0].scatter(train_scores, test_scores, alpha=0.5, color='forestgreen')
    axes[2, 0].plot([train_scores.min(), train_scores.max()],
                    [train_scores.min(), train_scores.max()],
                    'r--', lw=2, label='Linea perfecta')
    axes[2, 0].set_xlabel('Train F1-Score')
    axes[2, 0].set_ylabel('Test F1-Score')
    axes[2, 0].set_title('Analisis de Sobreajuste')
    axes[2, 0].legend()
    axes[2, 0].grid(True, alpha=0.3)

    # 6. Distribución de scores
    axes[2, 1].hist(test_scores, bins=20, alpha=0.7, label='Test F1-Scores',
                    edgecolor='black', color='forestgreen')
    axes[2, 1].axvline(dt_grid.best_score_, color='darkgreen', linestyle='--',
                       linewidth=2, label=f'Best: {dt_grid.best_score_:.4f}')
    axes[2, 1].set_xlabel('F1-Score')
    axes[2, 1].set_ylabel('Frecuencia')
    axes[2, 1].set_title('Distribucion de F1-Scores en RandomizedSearchCV')
    axes[2, 1].legend()
    axes[2, 1].grid(True, alpha=0.3)

    plt.tight_layout()

    # Guardar gráficas
    os.makedirs('tuning_results', exist_ok=True)
    plt.savefig('tuning_results/Decision_Tree_analysis.png', dpi=300, bbox_inches='tight')
    print("\n✓ Graficas guardadas en: tuning_results/Decision_Tree_analysis.png")
    plt.show()

    # =========================================================================
    # GUARDADO DE RESULTADOS
    # =========================================================================

    dt_results = {
        'model': dt_grid.best_estimator_,
        'model_name': 'Decision_Tree',
        'best_params': dt_grid.best_params_,
        'training_time': grid_time,
        'total_time': execution_time,
        'grid_search_results': dt_grid.cv_results_,
        'permutation_importance': perm_df,
        'native_importance': native_df,
        'depth_analysis': depth_grouped,
        'feature_names': list(feature_names),
        'cv_scores': {
            'mean': dt_cv.mean(),
            'std': dt_cv.std(),
            'scores': dt_cv
        },
        'metrics': {
            'train_f1_macro': train_f1,
            'test_f1_macro': test_f1,
            'train_accuracy': train_acc,
            'test_accuracy': test_acc,
            'train_balanced_accuracy': train_bal_acc,
            'test_balanced_accuracy': test_bal_acc,
            'train_precision_macro': train_precision,
            'test_precision_macro': test_precision,
            'train_recall_macro': train_recall,
            'test_recall_macro': test_recall
        },
        'predictions': {
            'y_train_pred': y_pred_train,
            'y_test_pred': y_pred_test
        },
        'confusion_matrix': cm,
        'classification_report': classification_report(y_test, y_pred_test),
        'overfitting_analysis': {
            'train_test_gap_mean': gap.mean(),
            'train_test_gap_std': gap.std()
        }
    }

    # joblib.dump(dt_results, 'tuning_results/Decision_Tree_results.pkl')
    # print("✓ Resultados completos guardados: tuning_results/Decision_Tree_results.pkl")

    joblib.dump(dt_grid.best_estimator_, 'tuning_results/best_Decision_Tree_model.pkl')
    print("✓ Mejor modelo guardado: tuning_results/best_Decision_Tree_model.pkl")

except Exception as e:
    print(f"\nERROR en Decision Tree: {e}")
    traceback.print_exc()
    dt_results = None


In [ ]:
# ============================================================================
# GAUSSIAN NAIVE BAYES - TUNING
# ============================================================================

print("\n" + "="*80)
print("MODELO: GAUSSIAN NAIVE BAYES")
print("="*80)

try:
    from sklearn.naive_bayes import GaussianNB
    from sklearn.inspection import permutation_importance
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score,
                                 precision_score, recall_score,
                                 classification_report, confusion_matrix)
    import numpy as np

    start_time = time.time()

    # Pipeline
    nb_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', GaussianNB())
    ])

    # Grid de hiperparámetros
    nb_param_grid = {
        'classifier__var_smoothing': np.logspace(-12, -3, 30)
    }

    print("\nHiperparametros a optimizar:")
    print("   var_smoothing: np.logspace(-12, -3, 30)")
    print(f"\nCombinaciones totales: 30\n")

    # GridSearch
    nb_grid = GridSearchCV(
        nb_pipeline,
        nb_param_grid,
        cv=cv_strategy,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=2,
        pre_dispatch='2*n_jobs',
        error_score='raise',
        return_train_score=True
    )

    print("Iniciando busqueda de hiperparametros...\n")
    grid_start = time.time()
    nb_grid.fit(X_train, y_train)
    grid_time = time.time() - grid_start

    print(f"\nEntrenamiento completado en {grid_time:.2f}s ({grid_time/60:.2f} min)")
    print(f"Mejor CV score (F1-Macro): {nb_grid.best_score_:.4f}")
    print(f"Mejores parametros: {nb_grid.best_params_}")

    # =========================================================================
    # ANÁLISIS DE COMPORTAMIENTO
    # =========================================================================

    print(f"\nANALIZANDO COMPORTAMIENTO DEL MODELO...")

    results_df = pd.DataFrame(nb_grid.cv_results_)
    train_scores = results_df['mean_train_score']
    test_scores = results_df['mean_test_score']
    gap = train_scores - test_scores

    print(f"\nAnalisis de sobreajuste:")
    print(f"  Diferencia promedio train-test: {gap.mean():.4f}")
    print(f"  Configuraciones probadas: {len(results_df)}")

    # =========================================================================
    # OBTENER FEATURE NAMES DEL PIPELINE ENTRENADO
    # =========================================================================

    print(f"\nOBTENIENDO NOMBRES DE FEATURES...")

    fitted_preprocessor = nb_grid.best_estimator_.named_steps['preprocessor']
    X_train_transformed = fitted_preprocessor.transform(X_train)
    X_test_transformed = fitted_preprocessor.transform(X_test)
    n_features_actual = X_train_transformed.shape[1]

    print(f"Dimensiones despues del preprocesamiento: {X_train_transformed.shape}")
    print(f"Numero real de features: {n_features_actual}")

    try:
        feature_names = fitted_preprocessor.get_feature_names_out()
        print(f"Feature names obtenidos automaticamente: {len(feature_names)}")
    except:
        feature_names = []
        for name, transformer, columns in fitted_preprocessor.transformers_:
            if name == 'num':
                feature_names.extend([f'num__{col}' for col in columns])
            elif name == 'cat':
                if hasattr(transformer.named_steps['onehot'], 'get_feature_names_out'):
                    cat_features = transformer.named_steps['onehot'].get_feature_names_out(columns)
                    feature_names.extend([f'cat__{feat}' for feat in cat_features])
        print(f"Feature names generados manualmente: {len(feature_names)}")

    if len(feature_names) != n_features_actual:
        print(f"ADVERTENCIA: Ajustando feature_names de {len(feature_names)} a {n_features_actual}")
        if len(feature_names) < n_features_actual:
            feature_names = list(feature_names) + [f'feature_{i}' for i in range(len(feature_names), n_features_actual)]
        else:
            feature_names = list(feature_names)[:n_features_actual]

    print(f"Total de features despues del ajuste: {len(feature_names)}")

    # =========================================================================
    # PERMUTATION IMPORTANCE
    # =========================================================================

    print(f"\nCALCULANDO IMPORTANCIA DE FEATURES...")
    perm_start = time.time()

    perm_result = permutation_importance(
        nb_grid.best_estimator_,
        X_test,
        y_test,
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scoring='f1_macro'
    )

    perm_time = time.time() - perm_start

    print(f"Dimensiones de importancias: {len(perm_result.importances_mean)}")
    print(f"Dimensiones de feature_names: {len(feature_names)}")

    if len(feature_names) != len(perm_result.importances_mean):
        print(f"ADVERTENCIA: Ajustando feature_names de {len(feature_names)} a {len(perm_result.importances_mean)}")
        if len(feature_names) < len(perm_result.importances_mean):
            feature_names = list(feature_names) + [f'feature_{i}' for i in range(len(feature_names), len(perm_result.importances_mean))]
        else:
            feature_names = list(feature_names)[:len(perm_result.importances_mean)]

    perm_df = pd.DataFrame({
        'feature': feature_names,
        'importance_mean': perm_result.importances_mean,
        'importance_std': perm_result.importances_std
    }).sort_values('importance_mean', ascending=False)

    print(f"Permutation Importance calculado en {perm_time:.2f}s")

    # =========================================================================
    # VALIDACIÓN CRUZADA ADICIONAL
    # =========================================================================

    print("\nEvaluacion adicional con validacion cruzada...")
    cv_start = time.time()
    nb_cv = cross_val_score(
        nb_grid.best_estimator_,
        X_train, y_train,
        cv=cv_strategy,
        scoring='f1_macro',
        n_jobs=-1
    )
    cv_time = time.time() - cv_start

    # =========================================================================
    # PREDICCIONES Y MÉTRICAS
    # =========================================================================

    print("Generando predicciones...")
    y_pred_train = nb_grid.best_estimator_.predict(X_train)
    y_pred_test = nb_grid.best_estimator_.predict(X_test)

    # Métricas completas
    train_f1 = f1_score(y_train, y_pred_train, average='macro')
    test_f1 = f1_score(y_test, y_pred_test, average='macro')
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    train_bal_acc = balanced_accuracy_score(y_train, y_pred_train)
    test_bal_acc = balanced_accuracy_score(y_test, y_pred_test)
    train_precision = precision_score(y_train, y_pred_train, average='macro')
    test_precision = precision_score(y_test, y_pred_test, average='macro')
    train_recall = recall_score(y_train, y_pred_train, average='macro')
    test_recall = recall_score(y_test, y_pred_test, average='macro')

    # =========================================================================
    # MOSTRAR RESUMEN CON FORMATO ESTANDARIZADO
    # =========================================================================

    end_time = time.time()
    execution_time = end_time - start_time

    # Preparar datos para print_model_summary
    metrics = {
        'train_f1_macro': train_f1,
        'test_f1_macro': test_f1,
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'train_balanced_accuracy': train_bal_acc,
        'test_balanced_accuracy': test_bal_acc,
        'train_precision_macro': train_precision,
        'test_precision_macro': test_precision,
        'train_recall_macro': train_recall,
        'test_recall_macro': test_recall
    }

    times = {
        'grid_time': grid_time,
        'perm_time': perm_time,
        'cv_time': cv_time,
        'total_time': execution_time
    }

    # Llamar a la función de resumen
    print_model_summary('Gaussian Naive Bayes', metrics, nb_grid.best_params_, times, nb_cv)

    # =========================================================================
    # TOP 10 FEATURES MÁS IMPORTANTES
    # =========================================================================

    print("\n📊 TOP 10 CARACTERÍSTICAS MÁS IMPORTANTES")
    print("-" * 100)
    top10_data = {
        'Rank': range(1, 11),
        'Feature': perm_df.head(10)['feature'].values,
        'Importancia': perm_df.head(10)['importance_mean'].values,
        'Desv. Est.': perm_df.head(10)['importance_std'].values
    }
    top10_df = pd.DataFrame(top10_data)
    print(top10_df.to_string(index=False))

    print(f"\n\n📊 REPORTE DE CLASIFICACIÓN DETALLADO")
    print("-" * 100)
    print(classification_report(y_test, y_pred_test))

    # =========================================================================
    # GRÁFICAS
    # =========================================================================

    print("\nGenerando graficas...")

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Gaussian Naive Bayes - Análisis Completo', fontsize=16, fontweight='bold')

    # 1. Top 15 Features más importantes
    top_features = perm_df.head(15)
    axes[0, 0].barh(range(len(top_features)), top_features['importance_mean'], color='mediumseagreen')
    axes[0, 0].set_yticks(range(len(top_features)))
    axes[0, 0].set_yticklabels(top_features['feature'], fontsize=8)
    axes[0, 0].set_xlabel('Importancia Media (F1-Score)')
    axes[0, 0].set_title('Top 15 Features - Permutation Importance')
    axes[0, 0].invert_yaxis()
    axes[0, 0].grid(True, alpha=0.3, axis='x')

    # 2. Matriz de Confusión
    cm = confusion_matrix(y_test, y_pred_test)
    classes = sorted(y_test.unique())
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[0, 1],
                xticklabels=classes, yticklabels=classes)
    axes[0, 1].set_title('Matriz de Confusion')
    axes[0, 1].set_ylabel('Real')
    axes[0, 1].set_xlabel('Prediccion')

    # 3. Train vs Test Scores (Overfitting Analysis)
    axes[1, 0].scatter(train_scores, test_scores, alpha=0.5, color='seagreen')
    axes[1, 0].plot([train_scores.min(), train_scores.max()],
                    [train_scores.min(), train_scores.max()],
                    'r--', lw=2, label='Linea perfecta')
    axes[1, 0].set_xlabel('Train F1-Score')
    axes[1, 0].set_ylabel('Test F1-Score')
    axes[1, 0].set_title('Analisis de Sobreajuste')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # 4. Distribución de scores
    axes[1, 1].hist(test_scores, bins=20, alpha=0.7, label='Test F1-Scores',
                    edgecolor='black', color='mediumseagreen')
    axes[1, 1].axvline(nb_grid.best_score_, color='darkgreen', linestyle='--',
                       linewidth=2, label=f'Best: {nb_grid.best_score_:.4f}')
    axes[1, 1].set_xlabel('F1-Score')
    axes[1, 1].set_ylabel('Frecuencia')
    axes[1, 1].set_title('Distribucion de F1-Scores en GridSearch')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()

    # Guardar gráficas
    os.makedirs('tuning_results', exist_ok=True)
    plt.savefig('tuning_results/Gaussian_Naive_Bayes_analysis.png', dpi=300, bbox_inches='tight')
    print("\n✓ Graficas guardadas en: tuning_results/Gaussian_Naive_Bayes_analysis.png")
    plt.show()

    # =========================================================================
    # GUARDADO DE RESULTADOS
    # =========================================================================

    nb_results = {
        'model': nb_grid.best_estimator_,
        'model_name': 'Gaussian_Naive_Bayes',
        'best_params': nb_grid.best_params_,
        'training_time': grid_time,
        'total_time': execution_time,
        'grid_search_results': nb_grid.cv_results_,
        'permutation_importance': perm_df,
        'feature_names': list(feature_names),
        'cv_scores': {
            'mean': nb_cv.mean(),
            'std': nb_cv.std(),
            'scores': nb_cv
        },
        'metrics': {
            'train_f1_macro': train_f1,
            'test_f1_macro': test_f1,
            'train_accuracy': train_acc,
            'test_accuracy': test_acc,
            'train_balanced_accuracy': train_bal_acc,
            'test_balanced_accuracy': test_bal_acc,
            'train_precision_macro': train_precision,
            'test_precision_macro': test_precision,
            'train_recall_macro': train_recall,
            'test_recall_macro': test_recall
        },
        'predictions': {
            'y_train_pred': y_pred_train,
            'y_test_pred': y_pred_test
        },
        'confusion_matrix': cm,
        'classification_report': classification_report(y_test, y_pred_test),
        'overfitting_analysis': {
            'train_test_gap_mean': gap.mean(),
            'train_test_gap_std': gap.std()
        }
    }

    # joblib.dump(nb_results, 'tuning_results/Gaussian_Naive_Bayes_results.pkl')
    # print("✓ Resultados completos guardados: tuning_results/Gaussian_Naive_Bayes_results.pkl")

    joblib.dump(nb_grid.best_estimator_, 'tuning_results/best_Gaussian_Naive_Bayes_model.pkl')
    print("✓ Mejor modelo guardado: tuning_results/best_Gaussian_Naive_Bayes_model.pkl")

except Exception as e:
    print(f"\nERROR en Gaussian Naive Bayes: {e}")
    traceback.print_exc()
    nb_results = None


In [ ]:
# ============================================================================
# LINEAR SVC - TUNING
# ============================================================================

print("\n" + "="*80)
print("MODELO 5/5: LINEAR SVC")
print("="*80)

try:
    from sklearn.svm import LinearSVC
    from sklearn.model_selection import GridSearchCV
    from sklearn.inspection import permutation_importance
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import (f1_score, accuracy_score, balanced_accuracy_score,
                                 precision_score, recall_score,
                                 classification_report, confusion_matrix)

    start_time = time.time()

    # Pipeline
    svc_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LinearSVC(random_state=RANDOM_STATE, max_iter=3000, dual='auto'))
    ])

    # Grid de parámetros - separado por penalty para evitar combinaciones inválidas
    # IMPORTANTE: LinearSVC no soporta penalty='l1' con loss='hinge'
    svc_param_grid = [
        # L2 penalty con ambos loss
        {
            'classifier__penalty': ['l2'],
            'classifier__loss': ['hinge', 'squared_hinge'],
            'classifier__C': [0.01, 0.1, 1, 10, 100],
            'classifier__class_weight': [None, 'balanced'],
            'classifier__fit_intercept': [True, False]
        },
        # L1 penalty solo con squared_hinge
        {
            'classifier__penalty': ['l1'],
            'classifier__loss': ['squared_hinge'],
            'classifier__C': [0.01, 0.1, 1, 10, 100],
            'classifier__class_weight': [None, 'balanced'],
            'classifier__fit_intercept': [True, False],
            'classifier__dual': [False]  # L1 requiere dual=False
        }
    ]

    total_combinations = (2 * 5 * 2 * 2) + (1 * 5 * 2 * 2)  # L2 + L1

    print("\nHiperparametros a optimizar:")
    print("   L2 penalty: loss=[hinge, squared_hinge], C=[0.01-100]")
    print("   L1 penalty: loss=[squared_hinge], C=[0.01-100], dual=False")
    print("   class_weight: [None, balanced]")
    print("   fit_intercept: [True, False]")
    print(f"\nCombinaciones totales: {total_combinations} (combinaciones válidas)\n")

    # GridSearch
    svc_grid = GridSearchCV(
        svc_pipeline,
        svc_param_grid,
        cv=cv_strategy,
        scoring='f1_macro',
        n_jobs=-1,
        verbose=2,
        return_train_score=True
    )

    print("Iniciando busqueda de hiperparametros...\n")
    grid_start = time.time()
    svc_grid.fit(X_train, y_train)
    grid_time = time.time() - grid_start

    print(f"\nEntrenamiento completado en {grid_time:.2f}s ({grid_time/60:.2f} min)")
    print(f"Mejor CV score (F1-Macro): {svc_grid.best_score_:.4f}")
    print(f"Mejores parametros: {svc_grid.best_params_}")

    # =========================================================================
    # ANÁLISIS DE COMPORTAMIENTO
    # =========================================================================

    print(f"\nANALIZANDO COMPORTAMIENTO DEL MODELO...")

    results_df = pd.DataFrame(svc_grid.cv_results_)
    train_scores = results_df['mean_train_score']
    test_scores = results_df['mean_test_score']
    gap = train_scores - test_scores

    print(f"\nAnalisis de sobreajuste:")
    print(f"  Diferencia promedio train-test: {gap.mean():.4f}")
    print(f"  Configuraciones probadas: {len(results_df)}")

    # =========================================================================
    # OBTENER FEATURE NAMES DEL PIPELINE ENTRENADO
    # =========================================================================

    print(f"\nOBTENIENDO NOMBRES DE FEATURES...")

    fitted_preprocessor = svc_grid.best_estimator_.named_steps['preprocessor']
    X_train_transformed = fitted_preprocessor.transform(X_train)
    X_test_transformed = fitted_preprocessor.transform(X_test)
    n_features_actual = X_train_transformed.shape[1]

    print(f"Dimensiones despues del preprocesamiento: {X_train_transformed.shape}")

    try:
        feature_names = fitted_preprocessor.get_feature_names_out()
    except:
        feature_names = [f'feature_{i}' for i in range(n_features_actual)]

    if len(feature_names) != n_features_actual:
        feature_names = list(feature_names)[:n_features_actual] if len(feature_names) > n_features_actual else \
                       list(feature_names) + [f'feature_{i}' for i in range(len(feature_names), n_features_actual)]

    # =========================================================================
    # PERMUTATION IMPORTANCE
    # =========================================================================

    print(f"\nCALCULANDO IMPORTANCIA DE FEATURES...")
    perm_start = time.time()

    perm_result = permutation_importance(
        svc_grid.best_estimator_,
        X_test,
        y_test,
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        scoring='f1_macro'
    )

    perm_time = time.time() - perm_start

    # Ajustar feature_names si es necesario
    if len(feature_names) != len(perm_result.importances_mean):
        if len(feature_names) < len(perm_result.importances_mean):
            feature_names = list(feature_names) + [f'feature_{i}' for i in range(len(feature_names), len(perm_result.importances_mean))]
        else:
            feature_names = list(feature_names)[:len(perm_result.importances_mean)]

    perm_df = pd.DataFrame({
        'feature': feature_names,
        'importance_mean': perm_result.importances_mean,
        'importance_std': perm_result.importances_std
    }).sort_values('importance_mean', ascending=False)

    print(f"Permutation Importance calculado en {perm_time:.2f}s")

    # =========================================================================
    # VALIDACIÓN CRUZADA ADICIONAL
    # =========================================================================

    print("\nEvaluacion adicional con validacion cruzada...")
    cv_start = time.time()
    svc_cv = cross_val_score(
        svc_grid.best_estimator_,
        X_train, y_train,
        cv=cv_strategy,
        scoring='f1_macro',
        n_jobs=-1
    )
    cv_time = time.time() - cv_start

    # =========================================================================
    # PREDICCIONES Y MÉTRICAS
    # =========================================================================

    print("Generando predicciones...")
    y_pred_train = svc_grid.best_estimator_.predict(X_train)
    y_pred_test = svc_grid.best_estimator_.predict(X_test)

    train_f1 = f1_score(y_train, y_pred_train, average='macro')
    test_f1 = f1_score(y_test, y_pred_test, average='macro')
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    train_bal_acc = balanced_accuracy_score(y_train, y_pred_train)
    test_bal_acc = balanced_accuracy_score(y_test, y_pred_test)
    train_precision = precision_score(y_train, y_pred_train, average='macro')
    test_precision = precision_score(y_test, y_pred_test, average='macro')
    train_recall = recall_score(y_train, y_pred_train, average='macro')
    test_recall = recall_score(y_test, y_pred_test, average='macro')

    # =========================================================================
    # MOSTRAR RESUMEN CON FORMATO ESTANDARIZADO
    # =========================================================================

    end_time = time.time()
    execution_time = end_time - start_time

    # Preparar datos para print_model_summary
    metrics = {
        'train_f1_macro': train_f1,
        'test_f1_macro': test_f1,
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'train_balanced_accuracy': train_bal_acc,
        'test_balanced_accuracy': test_bal_acc,
        'train_precision_macro': train_precision,
        'test_precision_macro': test_precision,
        'train_recall_macro': train_recall,
        'test_recall_macro': test_recall
    }

    times = {
        'grid_time': grid_time,
        'perm_time': perm_time,
        'cv_time': cv_time,
        'total_time': execution_time
    }

    # Llamar a la función de resumen
    print_model_summary('Linear SVC', metrics, svc_grid.best_params_, times, svc_cv)

    # =========================================================================
    # TOP 10 FEATURES MÁS IMPORTANTES
    # =========================================================================

    print("\n📊 TOP 10 CARACTERÍSTICAS MÁS IMPORTANTES")
    print("-" * 100)
    top10_data = {
        'Rank': range(1, 11),
        'Feature': perm_df.head(10)['feature'].values,
        'Importancia': perm_df.head(10)['importance_mean'].values,
        'Desv. Est.': perm_df.head(10)['importance_std'].values
    }
    top10_df = pd.DataFrame(top10_data)
    print(top10_df.to_string(index=False))

    print(f"\n\n📊 REPORTE DE CLASIFICACIÓN DETALLADO")
    print("-" * 100)
    print(classification_report(y_test, y_pred_test))

    # =========================================================================
    # GRÁFICAS
    # =========================================================================

    print("\nGenerando graficas...")

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Linear SVC - Análisis Completo', fontsize=16, fontweight='bold')

    # 1. Top 15 Features más importantes
    top_features = perm_df.head(15)
    axes[0, 0].barh(range(len(top_features)), top_features['importance_mean'], color='rebeccapurple')
    axes[0, 0].set_yticks(range(len(top_features)))
    axes[0, 0].set_yticklabels(top_features['feature'], fontsize=8)
    axes[0, 0].set_xlabel('Importancia Media (F1-Score)')
    axes[0, 0].set_title('Top 15 Features - Permutation Importance')
    axes[0, 0].invert_yaxis()
    axes[0, 0].grid(True, alpha=0.3, axis='x')

    # 2. Matriz de Confusión
    cm = confusion_matrix(y_test, y_pred_test)
    classes = sorted(y_test.unique())
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=axes[0, 1],
                xticklabels=classes, yticklabels=classes)
    axes[0, 1].set_title('Matriz de Confusion')
    axes[0, 1].set_ylabel('Real')
    axes[0, 1].set_xlabel('Prediccion')

    # 3. Train vs Test Scores
    axes[1, 0].scatter(train_scores, test_scores, alpha=0.5, color='indigo')
    axes[1, 0].plot([train_scores.min(), train_scores.max()],
                    [train_scores.min(), train_scores.max()],
                    'r--', lw=2, label='Linea perfecta')
    axes[1, 0].set_xlabel('Train F1-Score')
    axes[1, 0].set_ylabel('Test F1-Score')
    axes[1, 0].set_title('Analisis de Sobreajuste')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # 4. Distribución de scores
    axes[1, 1].hist(test_scores, bins=20, alpha=0.7, label='Test F1-Scores',
                    edgecolor='black', color='mediumpurple')
    axes[1, 1].axvline(svc_grid.best_score_, color='darkviolet', linestyle='--',
                       linewidth=2, label=f'Best: {svc_grid.best_score_:.4f}')
    axes[1, 1].set_xlabel('F1-Score')
    axes[1, 1].set_ylabel('Frecuencia')
    axes[1, 1].set_title('Distribucion de F1-Scores en GridSearch')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()

    # Guardar gráficas
    os.makedirs('tuning_results', exist_ok=True)
    plt.savefig('tuning_results/Linear_SVC_analysis.png', dpi=300, bbox_inches='tight')
    print("\n✓ Graficas guardadas en: tuning_results/Linear_SVC_analysis.png")
    plt.show()

    # =========================================================================
    # GUARDADO DE RESULTADOS
    # =========================================================================

    svm_results = {
        'model': svc_grid.best_estimator_,
        'model_name': 'Linear_SVC',
        'best_params': svc_grid.best_params_,
        'training_time': grid_time,
        'total_time': execution_time,
        'grid_search_results': svc_grid.cv_results_,
        'permutation_importance': perm_df,
        'feature_names': list(feature_names),
        'cv_scores': {
            'mean': svc_cv.mean(),
            'std': svc_cv.std(),
            'scores': svc_cv
        },
        'metrics': {
            'train_f1_macro': train_f1,
            'test_f1_macro': test_f1,
            'train_accuracy': train_acc,
            'test_accuracy': test_acc,
            'train_balanced_accuracy': train_bal_acc,
            'test_balanced_accuracy': test_bal_acc,
            'train_precision_macro': train_precision,
            'test_precision_macro': test_precision,
            'train_recall_macro': train_recall,
            'test_recall_macro': test_recall
        },
        'predictions': {
            'y_train_pred': y_pred_train,
            'y_test_pred': y_pred_test
        },
        'confusion_matrix': cm,
        'classification_report': classification_report(y_test, y_pred_test),
        'overfitting_analysis': {
            'train_test_gap_mean': gap.mean(),
            'train_test_gap_std': gap.std()
        }
    }

    # joblib.dump(svm_results, 'tuning_results/Linear_SVC_results.pkl')
    # print("✓ Resultados completos guardados: tuning_results/Linear_SVC_results.pkl")

    joblib.dump(svc_grid.best_estimator_, 'tuning_results/best_Linear_SVC_model.pkl')
    print("✓ Mejor modelo guardado: tuning_results/best_Linear_SVC_model.pkl")

except Exception as e:
    print(f"\nERROR en Linear SVC: {e}")
    traceback.print_exc()
    svm_results = None


In [ ]:
# ============================================================================
# TABLA COMPARATIVA - F1-SCORE PRIORITARIO
# ============================================================================

print("\n" + "="*100)
print("TABLA COMPARATIVA DE TODOS LOS MODELOS (F1-Score Prioritario)")
print("="*100)

# Recopilar resultados
all_results = []
model_names = {
    'lr_results': 'Logistic Regression',
    'ridge_results': 'Ridge Classifier',
    'dt_results': 'Decision Tree',
    'nb_results': 'Gaussian Naive Bayes',
    'svm_results': 'Linear SVC'
}

print("\nEstado de modelos:")
for var_name, display_name in model_names.items():
    if var_name in globals() and globals()[var_name] is not None:
        all_results.append(globals()[var_name])
        print(f"  ✓ {display_name}")
    else:
        print(f"  ✗ {display_name}: NO DISPONIBLE")

if len(all_results) == 0:
    print("\nERROR: No hay resultados para consolidar")
else:
    print(f"\nModelos completados: {len(all_results)}/5\n")

    # Crear DataFrame base
    data = {
        'Modelo': [],
        'F1_Test': [],
        'F1_Train': [],
        'F1_CV': [],
        'Overfitting': [],
        'Accuracy_Test': [],
        'Precision_Test': [],
        'Recall_Test': [],
        'Balanced_Acc_Test': [],
        'CV_Std': [],
        'Tiempo_min': []
    }

    # Extraer métricas de cada resultado
    for result in all_results:
        data['Modelo'].append(result['model_name'])

        # Métricas de test
        metrics = result.get('metrics', {})
        data['F1_Test'].append(metrics.get('test_f1_macro', 0))
        data['F1_Train'].append(metrics.get('train_f1_macro', 0))
        data['Accuracy_Test'].append(metrics.get('test_accuracy', 0))
        data['Precision_Test'].append(metrics.get('test_precision_macro', 0))
        data['Recall_Test'].append(metrics.get('test_recall_macro', 0))
        data['Balanced_Acc_Test'].append(metrics.get('test_balanced_accuracy', 0))

        # Validación cruzada
        cv_scores = result.get('cv_scores', {})
        data['F1_CV'].append(cv_scores.get('mean', 0))
        data['CV_Std'].append(cv_scores.get('std', 0))

        # Overfitting
        train_f1 = metrics.get('train_f1_macro', 0)
        test_f1 = metrics.get('test_f1_macro', 0)
        data['Overfitting'].append(train_f1 - test_f1)

        # Tiempo
        train_time = result.get('training_time', 0)
        data['Tiempo_min'].append(train_time / 60)

    # Crear DataFrame
    summary_df = pd.DataFrame(data)

    # Ordenar por F1_Test (métrica principal) de mayor a menor
    summary_df = summary_df.sort_values('F1_Test', ascending=False).reset_index(drop=True)
    summary_df.insert(0, 'Rank', range(1, len(summary_df) + 1))

    # =========================================================================
    # TABLA 1: MÉTRICAS PRINCIPALES (F1-Score enfocado)
    # =========================================================================

    print("\n" + "="*120)
    print("📊 TABLA 1: MÉTRICAS DE RENDIMIENTO (Ordenado por F1-Score Test)")
    print("="*120 + "\n")

    # Crear tabla simplificada con métricas principales
    main_metrics_df = summary_df[['Rank', 'Modelo', 'F1_Test', 'F1_Train', 'F1_CV',
                                   'Overfitting', 'Accuracy_Test', 'Tiempo_min']].copy()

    # Formatear para mejor visualización
    pd.set_option('display.float_format', lambda x: f'{x:.4f}')
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 150)

    print(main_metrics_df.to_string(index=False))

    # =========================================================================
    # TABLA 2: MÉTRICAS DETALLADAS
    # =========================================================================

    print("\n\n" + "="*120)
    print("📊 TABLA 2: MÉTRICAS DETALLADAS")
    print("="*120 + "\n")

    detailed_df = summary_df[['Rank', 'Modelo', 'Precision_Test', 'Recall_Test',
                              'Balanced_Acc_Test', 'CV_Std']].copy()
    print(detailed_df.to_string(index=False))

    # =========================================================================
    # TOP 3 MODELOS CON DETALLES
    # =========================================================================

    print("\n\n" + "="*120)
    print("🏆 TOP 3 MODELOS (Por F1-Score Test)")
    print("="*120)

    for i in range(min(3, len(summary_df))):
        row = summary_df.iloc(i)
        model_result = all_results[i]

        print(f"\n{'='*120}")
        print(f"#{i+1}. {row['Modelo'].upper()}")
        print(f"{'='*120}")

        # Métricas principales
        print(f"\n📈 MÉTRICAS PRINCIPALES:")
        print(f"   F1-Score Test:  {row['F1_Test']:.4f} ({row['F1_Test']*100:.2f}%)")
        print(f"   F1-Score Train: {row['F1_Train']:.4f} ({row['F1_Train']*100:.2f}%)")
        print(f"   F1-Score CV:    {row['F1_CV']:.4f} ± {row['CV_Std']:.4f}")

        # Status de overfitting
        overfitting_status = "✓ Buen ajuste" if abs(row['Overfitting']) < 0.05 else \
                           "⚠ Ligero sobreajuste" if abs(row['Overfitting']) < 0.10 else \
                           "✗ Sobreajuste significativo"
        print(f"   Overfitting:    {row['Overfitting']:+.4f} ({overfitting_status})")

        print(f"\n📊 OTRAS MÉTRICAS:")
        print(f"   Accuracy:       {row['Accuracy_Test']:.4f} ({row['Accuracy_Test']*100:.2f}%)")
        print(f"   Precision:      {row['Precision_Test']:.4f} ({row['Precision_Test']*100:.2f}%)")
        print(f"   Recall:         {row['Recall_Test']:.4f} ({row['Recall_Test']*100:.2f}%)")
        print(f"   Balanced Acc:   {row['Balanced_Acc_Test']:.4f} ({row['Balanced_Acc_Test']*100:.2f}%)")

        print(f"\n⏱  EFICIENCIA:")
        print(f"   Tiempo Total:   {row['Tiempo_min']:.2f} min")

        print(f"\n⚙️  MEJORES HIPERPARÁMETROS:")
        best_params = model_result.get('best_params', {})
        for param, value in best_params.items():
            param_clean = param.replace('classifier__', '')
            print(f"   {param_clean}: {value}")

    # =========================================================================
    # ANÁLISIS COMPARATIVO
    # =========================================================================

    print("\n\n" + "="*120)
    print("📊 ANÁLISIS COMPARATIVO")
    print("="*120)

    best_model = summary_df.iloc[0]
    worst_model = summary_df.iloc[-1]

    print(f"\n🥇 MEJOR MODELO: {best_model['Modelo']}")
    print(f"   F1-Score Test: {best_model['F1_Test']:.4f} ({best_model['F1_Test']*100:.2f}%)")
    print(f"   Mejora vs peor modelo: {(best_model['F1_Test'] - worst_model['F1_Test'])*100:.2f}%")

    print(f"\n📈 ESTADÍSTICAS GENERALES:")
    print(f"   F1-Score promedio: {summary_df['F1_Test'].mean():.4f}")
    print(f"   Desv. Est. F1:     {summary_df['F1_Test'].std():.4f}")
    print(f"   Rango F1:          [{summary_df['F1_Test'].min():.4f}, {summary_df['F1_Test'].max():.4f}]")

    print(f"\n⚠️  OVERFITTING:")
    avg_overfit = summary_df['Overfitting'].mean()
    print(f"   Overfitting promedio: {avg_overfit:+.4f}")
    print(f"   Modelos bien ajustados: {len(summary_df[summary_df['Overfitting'].abs() < 0.05])} de {len(summary_df)}")

    print(f"\n⏱  EFICIENCIA COMPUTACIONAL:")
    print(f"   Tiempo promedio:  {summary_df['Tiempo_min'].mean():.2f} min")
    print(f"   Modelo más rápido: {summary_df.loc[summary_df['Tiempo_min'].idxmin(), 'Modelo']} ({summary_df['Tiempo_min'].min():.2f} min)")
    print(f"   Modelo más lento:  {summary_df.loc[summary_df['Tiempo_min'].idxmax(), 'Modelo']} ({summary_df['Tiempo_min'].max():.2f} min)")

    # =========================================================================
    # TABLA DE HIPERPARÁMETROS
    # =========================================================================

    print("\n\n" + "="*120)
    print("⚙️  TABLA DE HIPERPARÁMETROS")
    print("="*120 + "\n")

    params_data = {'Modelo': data['Modelo']}

    # Recopilar todos los hiperparámetros únicos
    all_param_names = set()
    for result in all_results:
        best_params = result.get('best_params', {})
        for param in best_params.keys():
            param_clean = param.replace('classifier__', '')
            all_param_names.add(param_clean)

    # Crear columnas para cada hiperparámetro
    for param_name in sorted(all_param_names):
        params_data[param_name] = []
        for result in all_results:
            best_params = result.get('best_params', {})
            # Buscar el parámetro (con o sin prefijo)
            value = None
            for key, val in best_params.items():
                if key.replace('classifier__', '') == param_name:
                    value = val
                    break
            params_data[param_name].append(str(value) if value is not None else '-')

    params_df = pd.DataFrame(params_data)

    # Reordenar según ranking de F1-Score
    params_df['Rank'] = summary_df['Rank'].values
    params_df = params_df.sort_values('Rank')
    params_df = params_df.drop('Rank', axis=1)

    print(params_df.to_string(index=False))

    # =========================================================================
    # GUARDAR RESULTADOS
    # =========================================================================

    print("\n\n" + "="*120)
    print("💾 GUARDANDO RESULTADOS")
    print("="*120 + "\n")

    # CSV con todas las métricas
    summary_df.to_csv('modelos_restantes_comparacion_completa.csv', index=False)
    print("   ✓ modelos_restantes_comparacion_completa.csv")

    # CSV solo con métricas principales
    main_metrics_df.to_csv('modelos_restantes_metricas_principales.csv', index=False)
    print("   ✓ modelos_restantes_metricas_principales.csv")

    # CSV con métricas detalladas
    detailed_df.to_csv('modelos_restantes_metricas_detalladas.csv', index=False)
    print("   ✓ modelos_restantes_metricas_detalladas.csv")

    # CSV con hiperparámetros
    params_df.to_csv('modelos_restantes_hiperparametros.csv', index=False)
    print("   ✓ modelos_restantes_hiperparametros.csv")

    # Resultados completos en pickle
    joblib.dump(all_results, 'all_models_results.pkl', compress=3)
    print("   ✓ all_models_results.pkl")

    print("\n" + "="*120)
    print("✓ ANÁLISIS COMPARATIVO COMPLETADO")
    print("="*120)
